# UAV Fleet Strategy Engine with a Genetic Algorithm (Colab notebook)
**Artificial Intelligence – BSCS 5 · Assignment 2 · Bahria University Lahore Campus**

This notebook contains the **whole project, cell by cell**. Run the cells from top to bottom
(`Runtime → Run all`). Every code cell is the same code as the matching file in the
GitHub repository (`uav_ga/*.py` and `main.py`), so the notebook and the repository always agree.

| Step | Cell(s) | Matching file |
|---|---|---|
| 1 | Settings | `uav_ga/config.py` |
| 2 | The grid world | `uav_ga/environment.py` |
| 3 | Chromosome | `uav_ga/chromosome.py` |
| 4 | Fitness + energy model | `uav_ga/fitness.py` |
| 5 | Repair (constraints) | `uav_ga/constraints.py` |
| 6 | Selection / crossover / mutation | `uav_ga/operators.py` |
| 7 | GA engine | `uav_ga/ga.py` |
| 8 | Random-search baseline | `uav_ga/baseline.py` |
| 9 | Plots | `uav_ga/visualization.py` |
| 10 | Experiments | `main.py` |

Only `numpy`, `matplotlib` and `pandas` are needed – all are already installed on Colab.

In [ ]:
import time, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
print("numpy", np.__version__, "| pandas", pd.__version__)

## 1. Settings (`config.py`)
All numbers of the project live here. At the end of the cell we collect them into a small
object called `cfg`, so later cells can write `cfg.POP_SIZE`, exactly like the `.py` files do.
You can change a value later with e.g. `cfg.POP_SIZE = 50`.

In [ ]:
"""
All settings of the project in one place.

Every number used by the simulation, the fitness function and the Genetic
Algorithm (GA) is defined here as a plain Python variable. To change an
experiment you only edit this file (or override a value from the command line
in ``main.py``).

Units used everywhere:
    * distance on the grid  -> cells (1 cell = CELL_SIZE_M metres)
    * real distance         -> metres (m)
    * speed                 -> metres per second (m/s)
    * time                  -> seconds (s)
    * energy                -> joules (J)
"""

# ---------------------------------------------------------------------------
# 1. The world (grid) the drones fly over
# ---------------------------------------------------------------------------
GRID_SIZE = 100            # the map is GRID_SIZE x GRID_SIZE cells
CELL_SIZE_M = 10.0         # each cell is 10 m x 10 m, so the map is 1 km x 1 km
ENV_SEED = 7               # fixed seed -> every experiment uses the SAME map

N_NOFLY_ZONES = 7          # restricted airspace blocks (airport, buildings, ...)
N_WIND_POCKETS = 4         # Gaussian "high wind" hazard pockets
N_RADIATION_POCKETS = 3    # Gaussian "radiation" hazard pockets
N_PRIORITY_ZONES = 5       # disaster zones that we most want to survey
BACKGROUND_PRIORITY = 0.05 # every normal cell still has a small value
N_TARGETS = 30             # people / objects to detect on the ground

# Launch sites (x, y) in cell coordinates. The chromosome chooses one per drone.
BASE_STATIONS = [(6.5, 6.5), (93.5, 15.5), (48.5, 94.5)]

# ---------------------------------------------------------------------------
# 2. The drones (UAVs)
# ---------------------------------------------------------------------------
N_DRONES = 4
MAX_RANGE_M = 3000.0       # hard limit: a drone may fly at most 3 km
BATTERY_J = 60_000.0       # hard limit: 60 kJ (about 17 Wh) per drone
V_MIN = 3.0                # slowest allowed speed (m/s)
V_MAX = 18.0               # fastest allowed speed (m/s)
V_RETURN = 14.0            # speed used on the final "return to base" leg

SENSOR_RADIUS_MIN = 1.5    # sensor footprint radius (cells) at power 0
SENSOR_RADIUS_MAX = 5.0    # sensor footprint radius (cells) at power 1
SENSOR_POWER_MIN_W = 10.0  # electrical power of the sensor at power 0
SENSOR_POWER_MAX_W = 60.0  # electrical power of the sensor at power 1
WIND_POWER_FACTOR = 0.6    # flying in full wind costs 60 % more power
DETECTION_SPEED_FACTOR = 0.5  # at V_MAX the camera is 50 % less effective (motion blur)

# Rotary-wing power model of Zeng, Xu & Zhang (2019). These constants describe
# a typical small quadcopter. See fitness.flight_power() for the formula.
BLADE_PROFILE_POWER = 79.86   # P0 (W)
INDUCED_POWER = 88.63         # Pi (W)
ROTOR_TIP_SPEED = 120.0       # U_tip (m/s)
MEAN_ROTOR_VELOCITY = 4.03    # v0 (m/s)
FUSELAGE_DRAG_RATIO = 0.6     # d0
AIR_DENSITY = 1.225           # rho (kg/m^3)
ROTOR_SOLIDITY = 0.05         # s
ROTOR_DISC_AREA = 0.503       # A (m^2)

# ---------------------------------------------------------------------------
# 3. Chromosome shape
# ---------------------------------------------------------------------------
K_INIT_MIN = 6             # a new random drone gets 6..10 waypoints
K_INIT_MAX = 10
K_MIN = 2                  # mutation never deletes below 2 waypoints
K_MAX = 14                 # mutation never inserts above 14 waypoints
INIT_SENSOR_ON_PROB = 0.7  # chance that a random sensor bit starts as "on"

# ---------------------------------------------------------------------------
# 4. Fitness function
#    F = w1*Coverage + w2*Detection - w3*Energy - w4*Time - w5*Hazard - Penalty
# ---------------------------------------------------------------------------
W_COVERAGE = 0.35
W_DETECTION = 0.25
W_ENERGY = 0.15
W_TIME = 0.10
W_HAZARD = 0.15

# Constraint handling: "repair", "penalty" or "both"
CONSTRAINT_MODE = "both"
PENALTY_START = 2.0        # penalty multiplier at generation 0 ...
PENALTY_GROWTH = 4.0       # ... grows linearly to 2 * (1 + 4) = 10 at the last generation
NOFLY_WEIGHT = 10.0        # a no-fly incursion counts 10x more than a range overrun

# ---------------------------------------------------------------------------
# 5. Genetic Algorithm
# ---------------------------------------------------------------------------
SEED = 42
POP_SIZE = 100
GENERATIONS = 200
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.1        # probability that one gene is mutated
SELECTION = "tournament"   # "tournament", "rank" or "roulette"
TOURNAMENT_K = 3           # selection pressure of tournament selection
RANK_PRESSURE = 1.7        # selection pressure of linear rank selection (1..2)
ELITISM = 0.05             # top 5 % are copied unchanged to the next generation

BLX_ALPHA = 0.3            # how far BLX-alpha may go outside the parents
SEGMENT_SWAP_PROB = 0.5    # chance to take each drone segment from the other parent
SIGMA_START = 8.0          # Gaussian waypoint mutation step (cells) at the start ...
SIGMA_END = 1.0            # ... shrinks to this at the end (annealing)
RESET_PROB = 0.02          # small chance to re-create a whole drone at random

DIVERSITY_THRESHOLD = 0.3  # if diversity < 30 % of the starting diversity ...
IMMIGRANT_FRACTION = 0.1   # ... replace 10 % of the population with random newcomers
MUTATION_BOOST = 2.0       # ... and double the mutation rate for that generation

STAGNATION_PATIENCE = 60   # stop if the best fitness did not improve ...
STAGNATION_EPS = 1e-4      # ... by more than this for 60 generations
TARGET_FITNESS = None      # stop early if this fitness is reached (None = off)

# ---------------------------------------------------------------------------
# 6. Experiments
# ---------------------------------------------------------------------------
RESULTS_DIR = "results"
SWEEP_SEEDS = [1, 2, 3]            # seeds for each sensitivity setting
SWEEP_GENERATIONS = 100            # generations per sensitivity run
SWEEP_POP_SIZES = [20, 50, 100, 200]
SWEEP_MUTATION_RATES = [0.01, 0.05, 0.1, 0.2, 0.4]
SWEEP_TOURNAMENT_K = [2, 3, 5, 8]
SWEEP_SELECTIONS = ["roulette", "rank", "tournament"]
SWEEP_CONSTRAINT_MODES = ["penalty", "repair", "both"]
BASELINE_SEEDS = [1, 2, 3, 4, 5]   # seeds for GA vs. random search


def as_dict() -> dict:
    """Return every UPPER_CASE setting of this module as a dictionary.

    This is used to save the exact settings of a run next to its results,
    so every figure can be traced back to the numbers that produced it.
    """
    return {name: value for name, value in globals().items()
            if name.isupper() and not name.startswith("_")}


# --- notebook only: put every setting into one object called `cfg` ---
import types
cfg = types.SimpleNamespace(**{name: value for name, value in dict(globals()).items() if name.isupper()})
cfg.as_dict = lambda: {k: v for k, v in vars(cfg).items() if k.isupper()}
print("Grid:", cfg.GRID_SIZE, "x", cfg.GRID_SIZE, "| drones:", cfg.N_DRONES, "| population:", cfg.POP_SIZE)

## 2. The hazard grid (`environment.py`)
A 100 × 100 grid (1 cell = 10 m). Each cell is open or no-fly, has a hazard value (wind and
radiation pockets) and a priority value (disaster zones). Targets and base stations are also placed here.
It also contains the two search algorithms used by the repair step: **BFS** (nearest free cell) and **A\*** (detour path).

In [ ]:
"""
The 2-D grid world (the "hazard grid") that the drones survey.

The map is a GRID_SIZE x GRID_SIZE matrix. Each cell stores:
    * blocked  : True if the cell is a no-fly zone (value 1), else open (0)
    * hazard   : 0..1, how dangerous the cell is (wind + radiation pockets)
    * wind     : 0..1, the wind part of the hazard (it also costs energy)
    * priority : 0..1, how important it is to survey the cell (disaster zones)

Coordinate convention (used in the whole project):
    a point is (x, y) with 0 <= x, y < GRID_SIZE. The cell that contains the
    point is (int(x), int(y)). NumPy arrays are indexed as array[y, x]
    (row = y, column = x) so that ``plt.imshow(array, origin="lower")``
    draws the map the right way round.
"""

from collections import deque   # fast FIFO queue for Breadth-First Search
import heapq                     # priority queue (min-heap) for A* search

import numpy as np



def gaussian_blob(size: int, cx: float, cy: float, sigma: float) -> np.ndarray:
    """Return a size x size array with a 2-D Gaussian bump centred at (cx, cy).

    The peak value is 1 and it falls smoothly to 0 far away. We use this to
    model hazard pockets and disaster zones, which are strongest in the
    middle and weaker at the edges.
    """
    ys, xs = np.mgrid[0:size, 0:size] + 0.5          # (x, y) of every cell centre
    dist_sq = (xs - cx) ** 2 + (ys - cy) ** 2        # squared distance to the bump centre
    return np.exp(-dist_sq / (2.0 * sigma ** 2))     # Gaussian: 1 at the centre -> 0 far away


class Environment:
    """The survey area: no-fly zones, hazard field, priority map and targets.

    The map is generated from a fixed random seed (cfg.ENV_SEED), so every
    experiment and every GA run sees exactly the same world. Only the GA
    itself uses different seeds.
    """

    def __init__(self, seed: int = cfg.ENV_SEED, size: int = cfg.GRID_SIZE) -> None:
        self.size = size                                   # map is size x size cells
        self.cell_size_m = cfg.CELL_SIZE_M                 # metres per cell (to convert distances)
        self.base_stations = np.array(cfg.BASE_STATIONS, dtype=float)
        rng = np.random.default_rng(seed)                  # own generator -> same map every time

        # The ORDER of these calls matters: priority zones avoid no-fly zones,
        # radiation pockets are placed near priority zones, targets need both.
        self.blocked = self._make_nofly_zones(rng)
        self.priority_zones = []       # list of (cx, cy, sigma, weight) for plotting
        self.priority = self._make_priority_map(rng)
        self.wind, self.radiation = self._make_hazard_fields(rng)
        self.hazard = np.clip(self.wind + self.radiation, 0.0, 1.0)   # total danger, capped at 1
        self.targets = self._place_targets(rng)

        # Repair uses a slightly "inflated" no-fly mask (1 extra cell around
        # every zone) so repaired paths keep a safety margin from the border.
        self.blocked_margin = self._inflate(self.blocked, 1)
        self.nearest_free = self._nearest_free_cells()     # BFS lookup table, built once
        self.priority_total = float(self.priority.sum())   # denominator of the coverage objective
        self.cell_centres_x = np.arange(size) + 0.5   # used by the coverage code
        self._astar_cache = {}   # remembers A* answers: the same detour is asked for often

    # ------------------------------------------------------------------
    # Map generation
    # ------------------------------------------------------------------
    def _make_nofly_zones(self, rng: np.random.Generator) -> np.ndarray:
        """Place rectangles and circles of restricted airspace.

        Zones are not allowed near a base station, so drones can always take
        off. (For the default seed the remaining open airspace is one single
        connected region; a unit test checks this.)
        """
        n = self.size
        blocked = np.zeros((n, n), dtype=bool)             # start with a completely open map
        ys, xs = np.mgrid[0:n, 0:n] + 0.5                  # cell-centre coordinates
        placed = 0
        while placed < cfg.N_NOFLY_ZONES:
            cx, cy = rng.uniform(12, n - 12, size=2)       # zone centre, away from the map edge
            if rng.random() < 0.5:     # rectangle
                half_w, half_h = rng.uniform(3, 9, size=2)
                shape = (np.abs(xs - cx) <= half_w) & (np.abs(ys - cy) <= half_h)
            else:                      # circle
                radius = rng.uniform(4, 8)
                shape = (xs - cx) ** 2 + (ys - cy) ** 2 <= radius ** 2
            # the zone centre must be at least 21 cells (12 + largest half-size 9)
            # from every base station, so no zone can cover a launch site
            too_close = any(np.hypot(cx - bx, cy - by) < 12 + 9
                            for bx, by in self.base_stations)
            if too_close:
                continue                                   # reject and draw a new zone
            blocked |= shape                               # add the zone to the map (logical OR)
            placed += 1
        return blocked

    def _make_priority_map(self, rng: np.random.Generator) -> np.ndarray:
        """Disaster zones = Gaussian blobs with an importance weight 0.5..1."""
        n = self.size
        priority = np.full((n, n), cfg.BACKGROUND_PRIORITY)   # every cell has a small base value
        for _ in range(cfg.N_PRIORITY_ZONES):
            # pick a centre that is not inside a no-fly zone
            while True:
                cx, cy = rng.uniform(10, n - 10, size=2)
                if not self.blocked[int(cy), int(cx)]:
                    break
            sigma = rng.uniform(5, 10)                     # size of the disaster zone (cells)
            weight = rng.uniform(0.5, 1.0)                 # how important this zone is
            self.priority_zones.append((cx, cy, sigma, weight))
            # np.maximum: where zones overlap, keep the more important value
            priority = np.maximum(priority, weight * gaussian_blob(n, cx, cy, sigma))
        priority[self.blocked] = 0.0      # we never need to survey restricted airspace
        return priority

    def _make_hazard_fields(self, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
        """Wind and radiation pockets, each a sum of Gaussian bumps in 0..1.

        Two of the radiation pockets are placed close to disaster zones on
        purpose: in a real disaster the important area is often also the
        dangerous one, which creates a true trade-off for the GA.
        """
        n = self.size
        wind = np.zeros((n, n))
        for _ in range(cfg.N_WIND_POCKETS):
            cx, cy = rng.uniform(0, n, size=2)             # wind pockets can be anywhere
            # random strength 0.6..1.0 and random width 6..12 cells
            wind += rng.uniform(0.6, 1.0) * gaussian_blob(n, cx, cy, rng.uniform(6, 12))
        radiation = np.zeros((n, n))
        for i in range(cfg.N_RADIATION_POCKETS):
            if i < 2 and i < len(self.priority_zones):
                # first two pockets: near (+/- 6 cells) a disaster zone -> trade-off
                zx, zy, _, _ = self.priority_zones[i]
                cx, cy = zx + rng.uniform(-6, 6), zy + rng.uniform(-6, 6)
            else:
                cx, cy = rng.uniform(0, n, size=2)
            radiation += rng.uniform(0.6, 1.0) * gaussian_blob(n, cx, cy, rng.uniform(4, 8))
        return np.clip(wind, 0, 1), np.clip(radiation, 0, 1)   # keep both fields in 0..1

    def _place_targets(self, rng: np.random.Generator) -> np.ndarray:
        """Put targets mostly inside disaster zones (75 %) and some anywhere."""
        open_cells = np.argwhere(~self.blocked)            # rows of (y, x)
        # squaring the priority makes high-priority cells MUCH more likely
        weights = self.priority[open_cells[:, 0], open_cells[:, 1]] ** 2
        n_zone = int(round(0.75 * cfg.N_TARGETS))
        pick_zone = rng.choice(len(open_cells), size=n_zone, replace=False,
                               p=weights / weights.sum())  # weighted draw (inside zones)
        pick_any = rng.choice(len(open_cells), size=cfg.N_TARGETS - n_zone, replace=False)  # uniform draw
        cells = open_cells[np.concatenate([pick_zone, pick_any])]
        # convert (y, x) rows to (x, y) points in the middle of the cell
        return np.column_stack([cells[:, 1] + 0.5, cells[:, 0] + 0.5])

    @staticmethod
    def _inflate(mask: np.ndarray, margin: int) -> np.ndarray:
        """Grow every True region of ``mask`` by ``margin`` cells (8 directions)."""
        grown = mask.copy()
        # shift the mask one step in every direction (incl. diagonals) and OR it in
        for dy in range(-margin, margin + 1):
            for dx in range(-margin, margin + 1):
                shifted = np.roll(np.roll(mask, dy, axis=0), dx, axis=1)
                # np.roll wraps around the border; undo that wrap-around
                if dy > 0: shifted[:dy, :] = False
                if dy < 0: shifted[dy:, :] = False
                if dx > 0: shifted[:, :dx] = False
                if dx < 0: shifted[:, dx:] = False
                grown |= shifted
        return grown

    # ------------------------------------------------------------------
    # Search helpers used by the repair operator (BFS and A*)
    # ------------------------------------------------------------------
    def _nearest_free_cells(self) -> np.ndarray:
        """Multi-source Breadth-First Search from all free cells.

        Result: array[y, x] = (fx, fy), the closest free cell to (x, y).
        It is computed once, so moving a waypoint out of a no-fly zone later
        is just a table lookup.
        """
        n = self.size
        nearest = np.full((n, n, 2), -1, dtype=int)        # -1 means "not visited yet"
        queue = deque()
        # every free cell is a SOURCE of the search and is its own nearest free cell
        for y, x in np.argwhere(~self.blocked_margin):
            nearest[y, x] = (x, y)
            queue.append((x, y))
        # BFS spreads outwards one ring at a time, so the first source that
        # reaches a blocked cell is (one of) the closest free cells to it
        while queue:
            x, y = queue.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):   # 4 neighbours
                nx, ny = x + dx, y + dy
                if 0 <= nx < n and 0 <= ny < n and nearest[ny, nx, 0] < 0:
                    nearest[ny, nx] = nearest[y, x]   # inherit the source cell
                    queue.append((nx, ny))
        return nearest

    def astar(self, start: tuple[int, int], goal: tuple[int, int]) -> list[tuple[int, int]] | None:
        """A* search on the grid (8 neighbours) avoiding the inflated no-fly mask.

        Returns the list of cells from start to goal, or None if no path
        exists. The heuristic is the straight-line (Euclidean) distance,
        which never over-estimates, so the path found is the shortest one.
        """
        if (start, goal) in self._astar_cache:             # answered before? reuse it
            return self._astar_cache[(start, goal)]
        path = self._astar_search(start, goal)
        self._astar_cache[(start, goal)] = path
        return path

    def _astar_search(self, start: tuple[int, int], goal: tuple[int, int]) -> list[tuple[int, int]] | None:
        """The actual A* loop (called by astar(), which adds caching)."""
        n = self.size
        free = ~self.blocked_margin
        # (dx, dy, step cost): straight moves cost 1, diagonal moves cost sqrt(2)
        moves = [(1, 0, 1.0), (-1, 0, 1.0), (0, 1, 1.0), (0, -1, 1.0),
                 (1, 1, 1.414), (1, -1, 1.414), (-1, 1, 1.414), (-1, -1, 1.414)]
        open_heap = [(0.0, start)]         # (f = g + h, cell): always expand the smallest f first
        came_from = {start: None}          # parent of each cell, to rebuild the path at the end
        cost = {start: 0.0}                # g(cell): cheapest known cost from the start
        while open_heap:
            _, current = heapq.heappop(open_heap)
            if current == goal:
                # walk back through the parents to rebuild the path
                path = []
                while current is not None:
                    path.append(current)
                    current = came_from[current]
                return path[::-1]          # reverse: start -> goal
            cx, cy = current
            for dx, dy, step in moves:
                nx, ny = cx + dx, cy + dy
                if not (0 <= nx < n and 0 <= ny < n) or not free[ny, nx]:
                    continue               # outside the map or blocked
                # do not cut corners of a blocked cell when moving diagonally
                if dx and dy and (not free[cy, nx] or not free[ny, cx]):
                    continue
                new_cost = cost[current] + step
                if new_cost < cost.get((nx, ny), np.inf):    # found a cheaper way to this cell
                    cost[(nx, ny)] = new_cost
                    came_from[(nx, ny)] = current
                    h = np.hypot(goal[0] - nx, goal[1] - ny)  # heuristic: straight-line distance
                    heapq.heappush(open_heap, (new_cost + h, (nx, ny)))
        return None                        # queue empty: the goal cannot be reached

    # ------------------------------------------------------------------
    # Queries
    # ------------------------------------------------------------------
    def in_bounds(self, point: np.ndarray) -> bool:
        """True if the point lies on the map."""
        return 0 <= point[0] < self.size and 0 <= point[1] < self.size

    def is_free(self, point: np.ndarray, with_margin: bool = True) -> bool:
        """True if the cell containing ``point`` is open airspace."""
        mask = self.blocked_margin if with_margin else self.blocked
        return self.in_bounds(point) and not mask[int(point[1]), int(point[0])]   # note: [y, x]

    def segment_is_clear(self, a: np.ndarray, b: np.ndarray, with_margin: bool = True) -> bool:
        """True if the straight line a -> b never enters a no-fly cell.

        We sample points every 0.5 cells along the line (a simple and robust
        alternative to Bresenham's algorithm for real-valued end points).
        Because the margin mask makes every zone at least 3 cells wide, a
        0.5-cell step can never jump over a no-fly cell.
        """
        mask = self.blocked_margin if with_margin else self.blocked
        length = float(np.hypot(*(b - a)))                 # leg length in cells
        n_samples = max(2, int(np.ceil(length / 0.5)) + 1) # one sample every 0.5 cells
        t = np.linspace(0.0, 1.0, n_samples)               # 0 = point a, 1 = point b
        xs = np.clip((a[0] + t * (b[0] - a[0])).astype(int), 0, self.size - 1)
        ys = np.clip((a[1] + t * (b[1] - a[1])).astype(int), 0, self.size - 1)
        return not mask[ys, xs].any()                      # clear if NO sample is blocked

    def sample_legs(self, starts: np.ndarray, ends: np.ndarray, step: float = 0.5) -> dict:
        """Walk along many straight legs at once and read the map under them.

        Each leg i goes from starts[i] to ends[i]. Points are sampled every
        ``step`` cells. For every leg we return its length (cells), the mean
        wind and hazard under it, and how many samples were in no-fly cells.
        This is done with NumPy for all legs together because it is the most
        frequently called code in the whole GA.
        """
        vectors = ends - starts                            # direction of every leg
        lengths = np.hypot(vectors[:, 0], vectors[:, 1])   # length of every leg (cells)
        counts = np.maximum(1, np.ceil(lengths / step).astype(int))   # samples per leg
        # leg_id says which leg every sample belongs to, e.g. counts [2, 3] -> [0, 0, 1, 1, 1]
        leg_id = np.repeat(np.arange(len(starts)), counts)
        # local = sample number INSIDE its leg, e.g. [0, 1, 0, 1, 2]
        first_index = np.cumsum(counts) - counts
        local = np.arange(counts.sum()) - np.repeat(first_index, counts)
        # position of each sample inside its leg: 0.5/count, 1.5/count, ... (between 0 and 1)
        t = (local + 0.5) / counts[leg_id]
        px = starts[leg_id, 0] + t * vectors[leg_id, 0]    # x of every sample
        py = starts[leg_id, 1] + t * vectors[leg_id, 1]    # y of every sample
        xi = np.clip(px.astype(int), 0, self.size - 1)     # cell column
        yi = np.clip(py.astype(int), 0, self.size - 1)     # cell row
        # np.bincount(leg_id, values) adds up the values of each leg's samples
        return {
            "length": lengths,
            "mean_wind": np.bincount(leg_id, self.wind[yi, xi], len(starts)) / counts,
            "mean_hazard": np.bincount(leg_id, self.hazard[yi, xi], len(starts)) / counts,
            "blocked_samples": np.bincount(leg_id, self.blocked[yi, xi], len(starts)),
            "samples": counts,
        }

Let's build the map and look at it.

In [ ]:
env = Environment()
print("no-fly share of the map:", round(env.blocked.mean(), 3))
print("targets:", len(env.targets), "| base stations:", env.base_stations.tolist())
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, layer, title, cmap in zip(axes, [env.blocked, env.hazard, env.priority],
                                  ["No-fly zones (1 = blocked)", "Hazard field (0..1)", "Priority map (0..1)"],
                                  ["Greys", "Oranges", "Blues"]):
    im = ax.imshow(layer, origin="lower", cmap=cmap); ax.set_title(title); fig.colorbar(im, ax=ax)
plt.show()

## 3. The chromosome (`chromosome.py`)
One chromosome = the plan of the whole fleet. It has **one segment per drone**:
`[base | (x, y, speed, sensor power, sensor on/off) × K]` – a hybrid of integer, real and binary genes.

In [ ]:
"""
The chromosome: one complete mission plan for the whole drone fleet.

Encoding (hybrid, multi-segment)
--------------------------------
The chromosome has one SEGMENT per drone. A segment mixes integer, real and
binary genes, which is why it is called a "hybrid" encoding:

    segment = [ base | (x1, y1, v1, p1, a1) | (x2, y2, v2, p2, a2) | ... | (xK, yK, vK, pK, aK) ]

    base    integer  index of the launch site (0 .. number of bases - 1)
    x, y    real     waypoint position on the grid (cells)
    v       real     speed on the leg that ENDS at this waypoint (m/s)
    p       real     sensor power on that leg (0..1); bigger power = bigger footprint
    a       binary   1 = sensor switched on during that leg, 0 = off
                     (all the a-bits together form the sensor activation schedule)

The drone flies   base -> w1 -> w2 -> ... -> wK -> base.
The last "return home" leg always uses speed V_RETURN with the sensor off.
K (the number of waypoints) can be different for every drone and can change
during evolution (insert/delete mutation), so the chromosome has a variable
length. Inside the code each gene type is stored in its own NumPy array,
which keeps the operators simple and fast.
"""

import numpy as np



class DroneSegment:
    """The genes of ONE drone (one segment of the chromosome)."""

    def __init__(self, base: int, waypoints: np.ndarray, speeds: np.ndarray,
                 powers: np.ndarray, sensor_on: np.ndarray) -> None:
        # Each gene type gets its own array; index k always means "waypoint k",
        # so waypoints[k], speeds[k], powers[k] and sensor_on[k] belong together.
        self.base = int(base)                                      # integer gene
        self.waypoints = np.asarray(waypoints, dtype=float).reshape(-1, 2)  # K x 2 real genes
        self.speeds = np.asarray(speeds, dtype=float)              # K real genes
        self.powers = np.asarray(powers, dtype=float)              # K real genes
        self.sensor_on = np.asarray(sensor_on, dtype=bool)         # K binary genes

    @property
    def n_waypoints(self) -> int:
        """K, the number of waypoints of this drone (can change during evolution)."""
        return len(self.waypoints)

    def copy(self) -> "DroneSegment":
        """Independent copy: changing the copy never changes the original."""
        return DroneSegment(self.base, self.waypoints.copy(), self.speeds.copy(),
                            self.powers.copy(), self.sensor_on.copy())

    def route(self, env: Environment) -> np.ndarray:
        """All points the drone visits, including take-off and landing at its base."""
        base_xy = env.base_stations[self.base]                     # decode the integer gene
        return np.vstack([base_xy, self.waypoints, base_xy])      # base -> waypoints -> base

    @staticmethod
    def random(env: Environment, rng: np.random.Generator) -> "DroneSegment":
        """Create a random but valid drone segment.

        Waypoints are drawn from OPEN cells only, so a new random solution
        never starts inside a no-fly zone (legs may still cross one; the
        repair operator fixes that later).
        """
        k = int(rng.integers(cfg.K_INIT_MIN, cfg.K_INIT_MAX + 1))   # random number of waypoints
        open_cells = np.argwhere(~env.blocked_margin)              # rows of (y, x)
        chosen = open_cells[rng.integers(0, len(open_cells), size=k)]
        # swap (y, x) -> (x, y) and add a random offset inside the cell (0..1)
        waypoints = np.column_stack([chosen[:, 1], chosen[:, 0]]) + rng.random((k, 2))
        return DroneSegment(
            base=rng.integers(0, len(env.base_stations)),          # any base station
            waypoints=waypoints,
            speeds=rng.uniform(cfg.V_MIN, cfg.V_MAX, size=k),      # any allowed speed
            powers=rng.random(k),                                  # sensor power 0..1
            sensor_on=rng.random(k) < cfg.INIT_SENSOR_ON_PROB,     # about 70 % of legs "on"
        )


class Chromosome:
    """A full fleet plan = a list of DroneSegment objects (one per drone).

    After evaluation the chromosome also remembers its raw objective values,
    its constraint violation and its fitness, so we never evaluate the same
    unchanged individual twice (e.g. elites copied to the next generation).
    """

    def __init__(self, segments: list[DroneSegment]) -> None:
        self.segments = segments               # one DroneSegment per drone
        self.objectives: dict | None = None   # filled in by FitnessEvaluator
        self.violation: float = 0.0            # 0 = all hard constraints satisfied
        self.fitness: float = -np.inf          # -inf = "not evaluated yet"

    @staticmethod
    def random(env: Environment, rng: np.random.Generator) -> "Chromosome":
        """A random fleet plan: one random segment for each drone."""
        return Chromosome([DroneSegment.random(env, rng) for _ in range(cfg.N_DRONES)])

    def copy(self) -> "Chromosome":
        """Deep copy (genes + cached evaluation)."""
        clone = Chromosome([s.copy() for s in self.segments])
        clone.objectives = None if self.objectives is None else dict(self.objectives)
        clone.violation = self.violation
        clone.fitness = self.fitness
        return clone

    def forget_fitness(self) -> None:
        """Call after changing any gene: the old fitness is no longer valid."""
        self.objectives = None
        self.violation = 0.0
        self.fitness = -np.inf

    def n_genes(self) -> int:
        """Total number of genes (1 base + 5 genes per waypoint, per drone)."""
        return sum(1 + 5 * s.n_waypoints for s in self.segments)

    def is_within_bounds(self, env: Environment) -> bool:
        """Check every gene is inside its allowed range (used by the tests)."""
        for s in self.segments:
            ok = (0 <= s.base < len(env.base_stations)                                 # valid base index
                  and np.all(s.waypoints >= 0) and np.all(s.waypoints < env.size)      # on the map
                  and np.all(s.speeds >= cfg.V_MIN - 1e-9) and np.all(s.speeds <= cfg.V_MAX + 1e-9)
                  and np.all(s.powers >= 0) and np.all(s.powers <= 1)
                  and len(s.speeds) == len(s.powers) == len(s.sensor_on) == s.n_waypoints)  # arrays aligned
            if not ok:
                return False
        return True

    def describe(self) -> list[dict]:
        """A JSON-friendly summary of the plan (saved in run_summary.json)."""
        return [{
            "drone": i + 1,
            "base": s.base,
            "waypoints": np.round(s.waypoints, 2).tolist(),
            "speeds_mps": np.round(s.speeds, 2).tolist(),
            "sensor_power": np.round(s.powers, 2).tolist(),
            "sensor_on": s.sensor_on.astype(int).tolist(),
        } for i, s in enumerate(self.segments)]


def path_features(chrom: Chromosome, env: Environment, n_points: int = 10) -> np.ndarray:
    """Turn a chromosome into a fixed-length vector to measure diversity.

    Chromosomes have different lengths, so we cannot compare genes one by
    one. Instead each drone route is resampled to ``n_points`` points spaced
    evenly along the route, divided by the grid size (so the values are
    0..1), and followed by the drone's mean speed, power and sensor-on share.
    """
    parts = []
    for seg in chrom.segments:
        route = seg.route(env)
        step = np.hypot(*np.diff(route, axis=0).T)                 # length of every leg
        dist = np.concatenate([[0.0], np.cumsum(step)])            # distance travelled at each point
        where = np.linspace(0.0, dist[-1], n_points)               # n_points evenly spaced distances
        xs = np.interp(where, dist, route[:, 0]) / env.size        # x at those distances (0..1)
        ys = np.interp(where, dist, route[:, 1]) / env.size        # y at those distances (0..1)
        speed = (seg.speeds.mean() - cfg.V_MIN) / (cfg.V_MAX - cfg.V_MIN)   # mean speed scaled to 0..1
        parts.append(np.concatenate([xs, ys, [speed, seg.powers.mean(), seg.sensor_on.mean()]]))
    return np.concatenate(parts)                                   # 23 numbers per drone

In [ ]:
rng = np.random.default_rng(0)
example = Chromosome.random(env, rng)
for seg in example.segments:
    print(f"base {seg.base} | {seg.n_waypoints} waypoints | first waypoint {np.round(seg.waypoints[0], 1)}"
          f" | speeds {np.round(seg.speeds[:3], 1)}... | sensor bits {seg.sensor_on.astype(int)}")
print("total genes:", example.n_genes())

## 4. Fitness function and UAV energy model (`fitness.py`)
`F = 0.35·Coverage + 0.25·Detection − 0.15·Energy − 0.10·FlightTime − 0.15·Hazard − Penalty`
(all terms normalised to 0..1).

In [ ]:
"""
Flight simulation, UAV energy model and the multi-objective fitness function.

    F = w1*Coverage + w2*Detection - w3*Energy - w4*FlightTime - w5*Hazard - Penalty

Every term is normalised to 0..1 so that the weights w1..w5 really express
how important each goal is (otherwise a term measured in joules would
dominate a term measured as a fraction).

    Coverage    priority-weighted share of the map seen by an ACTIVE sensor
    Detection   expected share of targets detected
    Energy      fleet energy used / fleet battery capacity
    FlightTime  mission makespan (slowest drone) / longest possible flight
    Hazard      time-weighted average hazard intensity along all flights
    Penalty     dynamic penalty for constraint violations (see constraints.py)
"""

import numpy as np


# The longest a single feasible flight can last: max range at minimum speed.
# Dividing the makespan by this number puts the FlightTime objective in 0..1.
REFERENCE_TIME_S = cfg.MAX_RANGE_M / cfg.V_MIN


# ---------------------------------------------------------------------------
# UAV model
# ---------------------------------------------------------------------------
def flight_power(speed: np.ndarray) -> np.ndarray:
    """Propulsion power (W) of a rotary-wing drone flying at ``speed`` (m/s).

    Model of Zeng, Xu & Zhang (2019). It has three parts:
      * blade profile power - grows slowly with speed
      * induced power       - needed to stay in the air, DROPS as speed grows
      * parasite power      - air drag on the body, grows with speed^3
    Because of this U-shape, hovering or flying very slowly is NOT the
    cheapest option. With these constants the energy per metre (power / speed)
    keeps falling up to about 18 m/s, so inside our speed limits a faster
    drone always spends less energy per metre (but detects targets worse).
    """
    v = np.asarray(speed, dtype=float)
    # P0 * (1 + 3 v^2 / U_tip^2): power to turn the rotor blades through the air
    blade = cfg.BLADE_PROFILE_POWER * (1 + 3 * v ** 2 / cfg.ROTOR_TIP_SPEED ** 2)
    v0 = cfg.MEAN_ROTOR_VELOCITY
    # Pi * sqrt( sqrt(1 + v^4 / (4 v0^4)) - v^2 / (2 v0^2) ): power to create lift
    induced = cfg.INDUCED_POWER * np.sqrt(np.sqrt(1 + v ** 4 / (4 * v0 ** 4)) - v ** 2 / (2 * v0 ** 2))
    # 0.5 * d0 * rho * s * A * v^3: power lost to air drag on the body
    parasite = 0.5 * cfg.FUSELAGE_DRAG_RATIO * cfg.AIR_DENSITY * cfg.ROTOR_SOLIDITY * cfg.ROTOR_DISC_AREA * v ** 3
    return blade + induced + parasite


def sensor_power_w(power_gene: np.ndarray) -> np.ndarray:
    """Electrical power drawn by the sensor for a power gene in 0..1."""
    # linear between 10 W (gene = 0) and 60 W (gene = 1)
    return cfg.SENSOR_POWER_MIN_W + (cfg.SENSOR_POWER_MAX_W - cfg.SENSOR_POWER_MIN_W) * power_gene


def sensor_radius(power_gene: np.ndarray) -> np.ndarray:
    """Footprint radius in cells: a stronger sensor sees further."""
    # linear between 1.5 cells (gene = 0) and 5 cells (gene = 1)
    return cfg.SENSOR_RADIUS_MIN + (cfg.SENSOR_RADIUS_MAX - cfg.SENSOR_RADIUS_MIN) * power_gene


# ---------------------------------------------------------------------------
# Flight simulation
# ---------------------------------------------------------------------------
def build_legs(chrom: Chromosome, env: Environment) -> dict:
    """Flatten the whole fleet into one table of straight legs.

    Leg k of a drone ends at waypoint k and uses that waypoint's speed,
    power and sensor bit. The extra last leg returns to base at V_RETURN
    with the sensor off.
    """
    starts, ends, speeds, powers, active, drone = [], [], [], [], [], []
    for d, seg in enumerate(chrom.segments):
        route = seg.route(env)                             # base, w1, ..., wK, base
        starts.append(route[:-1])                          # every point except the last
        ends.append(route[1:])                             # every point except the first
        speeds.append(np.append(seg.speeds, cfg.V_RETURN)) # + speed of the return leg
        powers.append(np.append(seg.powers, 0.0))          # + return leg: no sensor power
        active.append(np.append(seg.sensor_on, False))     # + return leg: sensor off
        drone.append(np.full(len(route) - 1, d))           # remember which drone flies each leg
    # stack the lists of all drones into single NumPy arrays (one row per leg)
    return {"start": np.vstack(starts), "end": np.vstack(ends),
            "speed": np.concatenate(speeds), "power": np.concatenate(powers),
            "active": np.concatenate(active), "drone": np.concatenate(drone)}


def simulate_flight(chrom: Chromosome, env: Environment) -> dict:
    """Fly every leg and compute distance, time, energy and hazard dose.

    Returns per-drone totals (arrays of length N_DRONES) plus the leg table.
    """
    legs = build_legs(chrom, env)
    probe = env.sample_legs(legs["start"], legs["end"])    # read the map under every leg
    length_m = probe["length"] * env.cell_size_m           # cells -> metres
    time_s = length_m / legs["speed"]                      # time = distance / speed
    # wind makes the motors work harder: power x (1 + 0.6 * mean wind)
    propulsion_w = flight_power(legs["speed"]) * (1 + cfg.WIND_POWER_FACTOR * probe["mean_wind"])
    sensor_w = np.where(legs["active"], sensor_power_w(legs["power"]), 0.0)   # 0 W if sensor off
    energy_j = (propulsion_w + sensor_w) * time_s          # energy = power x time
    hazard_dose = probe["mean_hazard"] * time_s   # intensity x seconds spent

    n = len(chrom.segments)
    # add up the leg values of each drone (legs["drone"] says which drone a leg belongs to)
    per_drone = lambda values: np.bincount(legs["drone"], values, minlength=n)
    return {
        "legs": legs,
        "leg_length_m": length_m,
        "leg_energy_j": energy_j,
        "length_m": per_drone(length_m),
        "time_s": per_drone(time_s),
        "energy_j": per_drone(energy_j),
        "hazard_dose": per_drone(hazard_dose),
        "blocked_samples": int(probe["blocked_samples"].sum()),   # samples inside no-fly zones
        "total_samples": int(probe["samples"].sum()),
    }


def point_to_segment_distance(px, py, ax, ay, bx, by):
    """Shortest distance from point(s) (px, py) to the segment A-B.

    We project the point onto the line, clamp the projection to the segment
    (t in 0..1) and measure the distance to that closest point.
    """
    abx, aby = bx - ax, by - ay                            # vector A -> B
    denom = abx * abx + aby * aby + 1e-12                  # |AB|^2 (+ tiny value to avoid / 0)
    t = np.clip(((px - ax) * abx + (py - ay) * aby) / denom, 0.0, 1.0)   # projection, kept on the segment
    return np.hypot(px - (ax + t * abx), py - (ay + t * aby))           # distance to the closest point


def coverage_map(chrom: Chromosome, env: Environment, legs: dict | None = None) -> np.ndarray:
    """Boolean map[y, x]: True where some active sensor footprint covered the cell.

    A cell is covered if its centre is within the sensor radius of a leg
    flown with the sensor ON. To save time we only check the cells inside
    the bounding box of each leg (grown by the radius).
    """
    if legs is None:
        legs = build_legs(chrom, env)
    covered = np.zeros((env.size, env.size), dtype=bool)   # nothing covered yet
    centres = env.cell_centres_x
    for i in np.flatnonzero(legs["active"]):               # only legs with the sensor ON
        (ax, ay), (bx, by) = legs["start"][i], legs["end"][i]
        r = float(sensor_radius(legs["power"][i]))
        # bounding box of the leg, grown by the radius and cut to the map
        x0, x1 = int(max(0, min(ax, bx) - r)), int(min(env.size, max(ax, bx) + r + 1))
        y0, y1 = int(max(0, min(ay, by) - r)), int(min(env.size, max(ay, by) + r + 1))
        px = centres[x0:x1][None, :]                       # row vector of x centres
        py = centres[y0:y1][:, None]                       # column vector of y centres -> broadcasting
        covered[y0:y1, x0:x1] |= point_to_segment_distance(px, py, ax, ay, bx, by) <= r
    return covered


def detection_probabilities(legs: dict, env: Environment) -> np.ndarray:
    """Probability that each target is detected by at least one active leg.

    For one leg:  P = quality(power) * speed_factor(speed) * max(0, 1 - (d / r)^2)
      * quality      0.4 .. 1.0, a stronger sensor is more reliable
      * speed_factor 1.0 at V_MIN .. 0.5 at V_MAX (motion blur)
      * distance     targets near the centre line are easier to spot
    Several legs act like independent tries:  P_total = 1 - prod(1 - P_leg).
    """
    active = legs["active"]
    if not active.any():                                   # all sensors off -> nothing detected
        return np.zeros(len(env.targets))
    a, b = legs["start"][active], legs["end"][active]
    power, speed = legs["power"][active], legs["speed"][active]
    # table of distances: one row per active leg, one column per target
    tx, ty = env.targets[:, 0][None, :], env.targets[:, 1][None, :]
    dist = point_to_segment_distance(tx, ty, a[:, :1], a[:, 1:], b[:, :1], b[:, 1:])
    radius = sensor_radius(power)[:, None]
    quality = (0.4 + 0.6 * power)[:, None]
    speed_factor = (1 - cfg.DETECTION_SPEED_FACTOR * (speed - cfg.V_MIN) / (cfg.V_MAX - cfg.V_MIN))[:, None]
    p_leg = quality * speed_factor * np.clip(1 - (dist / radius) ** 2, 0.0, 1.0)
    # probability that at least one leg detects the target = 1 - P(every leg misses it)
    return 1.0 - np.prod(1.0 - p_leg, axis=0)


# ---------------------------------------------------------------------------
# Fitness evaluator
# ---------------------------------------------------------------------------
class FitnessEvaluator:
    """Computes the objectives and the scalar fitness of chromosomes.

    It also counts how many evaluations were made, so the random-search
    baseline can be given exactly the same budget as the GA.
    """

    def __init__(self, env: Environment, total_generations: int = cfg.GENERATIONS,
                 constraint_mode: str = cfg.CONSTRAINT_MODE) -> None:
        self.env = env
        self.total_generations = max(1, total_generations)   # used by the dynamic penalty
        self.constraint_mode = constraint_mode
        self.n_evaluations = 0                                # how many plans were simulated
        # same order as the objectives in score(): coverage, detection, energy, time, hazard
        self.weights = np.array([cfg.W_COVERAGE, cfg.W_DETECTION, cfg.W_ENERGY, cfg.W_TIME, cfg.W_HAZARD])

    def objectives(self, chrom: Chromosome) -> dict:
        """Simulate the plan and return all raw and normalised objective values."""
        flight = simulate_flight(chrom, self.env)
        legs = flight["legs"]
        covered = coverage_map(chrom, self.env, legs)
        p_detect = detection_probabilities(legs, self.env)
        n = len(chrom.segments)
        total_time = flight["time_s"].sum()

        # --- constraint violations (0 means feasible) ---
        # share of the flown path that lies inside no-fly zones
        nofly_fraction = flight["blocked_samples"] / max(1, flight["total_samples"])
        # how far each drone goes over its range / battery, as a fraction (0.1 = 10 % over)
        range_overrun = np.maximum(0, flight["length_m"] / cfg.MAX_RANGE_M - 1).sum() / n
        battery_overrun = np.maximum(0, flight["energy_j"] / cfg.BATTERY_J - 1).sum() / n

        high_priority = self.env.priority >= 0.5           # cells inside the disaster zones
        return {
            # normalised objectives (0..1)
            "coverage": float((self.env.priority * covered).sum() / self.env.priority_total),
            "detection": float(p_detect.mean()),
            "energy": float(flight["energy_j"].sum() / (n * cfg.BATTERY_J)),
            "time": float(flight["time_s"].max() / REFERENCE_TIME_S),     # slowest drone decides
            "hazard": float(flight["hazard_dose"].sum() / max(total_time, 1e-9)),
            # raw values, logged for the report
            "high_priority_coverage": float(covered[high_priority].mean()),
            "area_covered_frac": float(covered.mean()),
            "targets_detected": int((p_detect >= 0.5).sum()),
            "energy_kj": float(flight["energy_j"].sum() / 1000),
            "makespan_s": float(flight["time_s"].max()),
            "distance_km": float(flight["length_m"].sum() / 1000),
            "nofly_fraction": float(nofly_fraction),
            "range_overrun": float(range_overrun),
            "battery_overrun": float(battery_overrun),
            "n_waypoints": int(sum(s.n_waypoints for s in chrom.segments)),
        }

    def penalty_factor(self, generation: int) -> float:
        """Dynamic penalty multiplier: small early (explore), large late (be feasible).

        lambda(g) = PENALTY_START * (1 + PENALTY_GROWTH * g / G)
        In pure "repair" mode the penalty is switched off (lambda = 0).
        """
        if self.constraint_mode == "repair":
            return 0.0
        progress = min(1.0, generation / self.total_generations)    # 0 at the start, 1 at the end
        return cfg.PENALTY_START * (1 + cfg.PENALTY_GROWTH * progress)

    @staticmethod
    def violation(obj: dict) -> float:
        """One number that measures how badly the hard constraints are broken."""
        return cfg.NOFLY_WEIGHT * obj["nofly_fraction"] + obj["range_overrun"] + obj["battery_overrun"]

    def score(self, obj: dict, generation: int) -> float:
        """Turn the objective values into the single fitness number F."""
        terms = np.array([obj["coverage"], obj["detection"], obj["energy"], obj["time"], obj["hazard"]])
        signs = np.array([1, 1, -1, -1, -1])               # + maximise, - minimise
        weighted = float(np.sum(signs * self.weights * terms))
        return weighted - self.penalty_factor(generation) * self.violation(obj)

    def evaluate(self, chrom: Chromosome, generation: int) -> float:
        """Evaluate a chromosome (only if it is new) and store the results on it."""
        if chrom.objectives is None:                       # never simulated -> simulate now
            chrom.objectives = self.objectives(chrom)
            self.n_evaluations += 1                        # only REAL simulations are counted
        # the penalty changes with the generation, so the score is always recomputed (cheap)
        chrom.violation = self.violation(chrom.objectives)
        chrom.fitness = self.score(chrom.objectives, generation)
        return chrom.fitness

## 5. Constraint handling – repair (`constraints.py`)

In [ ]:
"""
Constraint handling: the REPAIR operator.

Hard constraints of the problem:
    1. every gene must be inside its range (grid bounds, speed limits, ...)
    2. no waypoint and no flight leg may enter a no-fly zone
    3. each drone must stay within MAX_RANGE_M and its BATTERY_J

Two ways to handle them are implemented (chosen with cfg.CONSTRAINT_MODE):
    * "penalty" - only clip genes to their bounds; every remaining violation
                  is punished by the dynamic penalty in fitness.py
    * "repair"  - actively fix the chromosome (this file), no penalty
    * "both"    - repair first, then penalise anything repair could not fix

The repaired genes are written back into the chromosome ("Lamarckian"
repair), so children inherit the feasible version of their parents.
"""

import numpy as np



def clip_genes(chrom: Chromosome, env: Environment) -> None:
    """Constraint 1: force every gene back into its allowed range."""
    for seg in chrom.segments:
        seg.base = int(np.clip(seg.base, 0, len(env.base_stations) - 1))     # valid base index
        np.clip(seg.waypoints, 0.0, env.size - 1e-6, out=seg.waypoints)       # stay on the map
        np.clip(seg.speeds, cfg.V_MIN, cfg.V_MAX, out=seg.speeds)             # legal speeds
        np.clip(seg.powers, 0.0, 1.0, out=seg.powers)                         # power 0..1
        # (sensor bits are True/False, so they can never be out of range)


def move_waypoints_out_of_nofly(seg: DroneSegment, env: Environment) -> None:
    """Constraint 2a: a waypoint inside a no-fly zone jumps to the nearest free cell.

    The nearest free cell of every cell was found once with a BFS when the
    environment was built (env.nearest_free), so this is a quick lookup.
    """
    for k, (x, y) in enumerate(seg.waypoints):
        if not env.is_free(np.array([x, y])):
            fx, fy = env.nearest_free[int(y), int(x)]      # BFS lookup table
            seg.waypoints[k] = (fx + 0.5, fy + 0.5)        # move to the centre of that free cell


def find_detour(a: np.ndarray, b: np.ndarray, env: Environment, depth: int = 0) -> list[np.ndarray] | None:
    """Return extra points that lead from a to b around the no-fly zones.

    Step 1 (cheap): try ONE extra point to the left or right of the middle of
            the leg, moving further out (3, 6, 9 ... cells) until both new
            legs are clear.
    Step 2 (divide and conquer): if no single point works, take the closest
            free side point that clears at least one half of the leg and
            solve the two halves again with the same method (max 3 levels).
    Step 3 (fallback): run A* on the grid, then remove unnecessary points
            ("string pulling": keep jumping to the furthest point that can be
            reached in a straight line).
    Returns [] if the leg is already clear and None if no detour exists.
    """
    if env.segment_is_clear(a, b):
        return []                                          # nothing to fix
    direction = b - a
    length = float(np.hypot(*direction))
    if length > 1e-9:
        normal = np.array([-direction[1], direction[0]]) / length   # 90 degrees to the leg
        middle = (a + b) / 2
        half_way_point = None                              # best candidate for step 2
        for offset in range(3, 46, 3):                     # 3, 6, ..., 45 cells away from the leg
            for side in (1, -1):                           # try the left side, then the right side
                c = middle + side * offset * normal
                if not env.is_free(c):
                    continue                               # candidate itself is blocked / off the map
                first_ok, second_ok = env.segment_is_clear(a, c), env.segment_is_clear(c, b)
                if first_ok and second_ok:
                    return [c]                                       # step 1 worked
                if half_way_point is None and (first_ok or second_ok):
                    half_way_point = c                     # remember the first "half solution"
        if half_way_point is not None and depth < 3:                 # step 2
            # solve a -> half_way_point and half_way_point -> b separately (recursion)
            first = find_detour(a, half_way_point, env, depth + 1)
            second = find_detour(half_way_point, b, env, depth + 1)
            if first is not None and second is not None:
                return first + [half_way_point] + second

    cells = env.astar((int(a[0]), int(a[1])), (int(b[0]), int(b[1])))   # step 3
    if cells is None:
        return None                                        # truly unreachable: leave it to the penalty
    # turn the A* cells into points (cell centres), with the exact end points a and b
    points = [a] + [np.array([cx + 0.5, cy + 0.5]) for cx, cy in cells[1:-1]] + [b]
    # string pulling: from point i jump to the FURTHEST point j that is visible in a straight line
    kept, i = [], 0
    while i < len(points) - 1:
        j = len(points) - 1
        while j > i + 1 and not env.segment_is_clear(points[i], points[j]):
            j -= 1
        kept.append(points[j])
        i = j
    return kept[:-1]          # the last kept point is b itself


def fix_crossing_legs(seg: DroneSegment, env: Environment) -> None:
    """Constraint 2b: insert detour waypoints into legs that cross a no-fly zone.

    A detour point inherits the speed / power / sensor bit of the leg it
    splits, so the drone's sensor schedule does not change. Detour points
    on the return-to-base leg use the return speed with the sensor off.
    """
    base_xy = env.base_stations[seg.base]
    new_wp, new_v, new_p, new_on = [], [], [], []          # the rebuilt gene lists
    previous = base_xy                                     # the drone starts at its base
    for k in range(seg.n_waypoints + 1):            # +1 = the return leg
        is_return = k == seg.n_waypoints
        target = base_xy if is_return else seg.waypoints[k]
        speed = cfg.V_RETURN if is_return else seg.speeds[k]
        power = 0.0 if is_return else seg.powers[k]
        sensor = False if is_return else seg.sensor_on[k]
        # 1) detour points (if the leg previous -> target crosses a no-fly zone)
        for point in find_detour(previous, target, env) or []:
            new_wp.append(point)
            new_v.append(speed)
            new_p.append(power)
            new_on.append(sensor)
        # 2) the original waypoint itself (the base is not stored as a waypoint)
        if not is_return:
            new_wp.append(target)
            new_v.append(speed)
            new_p.append(power)
            new_on.append(sensor)
        previous = target
    # write the repaired genes back into the chromosome (Lamarckian repair)
    seg.waypoints = np.array(new_wp).reshape(-1, 2)
    seg.speeds, seg.powers, seg.sensor_on = np.array(new_v), np.array(new_p), np.array(new_on, dtype=bool)


def trim_to_limits(seg: DroneSegment, env: Environment) -> None:
    """Constraint 3: drop waypoints from the END of the route until the drone
    can fly the route AND come back home within its range and battery.

    1. If the route is already within the limits, nothing happens.
    2. Otherwise we ESTIMATE how many waypoints can be kept: for every k we
       add the cost of flying to waypoint k and a pessimistic straight flight
       home (full wind). This avoids re-simulating the route k times.
    3. We then check the shortened route exactly (the way home may need a
       detour around a no-fly zone) and drop one more waypoint if needed.
    """
    flight = simulate_flight(Chromosome([seg]), env)       # simulate this one drone
    if flight["length_m"][0] <= cfg.MAX_RANGE_M and flight["energy_j"][0] <= cfg.BATTERY_J:
        return                                             # already feasible
    base_xy = env.base_stations[seg.base]
    distance_so_far = np.cumsum(flight["leg_length_m"][:-1])     # to waypoint 1, 2, ..., K
    energy_so_far = np.cumsum(flight["leg_energy_j"][:-1])
    # straight flight home from every waypoint, with worst-case wind (factor 1 + 0.6)
    home_m = np.hypot(*(seg.waypoints - base_xy).T) * env.cell_size_m
    home_j = home_m / cfg.V_RETURN * flight_power(cfg.V_RETURN) * (1 + cfg.WIND_POWER_FACTOR)
    fits = (distance_so_far + home_m <= cfg.MAX_RANGE_M) & (energy_so_far + home_j <= cfg.BATTERY_J)
    keep = int(np.flatnonzero(fits).max()) + 1 if fits.any() else 1   # last waypoint that still fits

    while True:
        # the route shortened to its first `keep` waypoints
        trial = DroneSegment(seg.base, seg.waypoints[:keep], seg.speeds[:keep],
                             seg.powers[:keep], seg.sensor_on[:keep])
        # the new way home may cross a no-fly zone -> add a detour (return speed, sensor off)
        for point in find_detour(trial.waypoints[-1], base_xy, env) or []:
            trial.waypoints = np.vstack([trial.waypoints, point])
            trial.speeds = np.append(trial.speeds, cfg.V_RETURN)
            trial.powers = np.append(trial.powers, 0.0)
            trial.sensor_on = np.append(trial.sensor_on, False)
        check = simulate_flight(Chromosome([trial]), env)  # exact check of the shortened route
        if (check["length_m"][0] <= cfg.MAX_RANGE_M and check["energy_j"][0] <= cfg.BATTERY_J) or keep <= 1:
            break                                          # feasible (or nothing left to drop)
        keep -= 1                                          # still too long: drop one more waypoint
    seg.waypoints, seg.speeds, seg.powers, seg.sensor_on = (
        trial.waypoints, trial.speeds, trial.powers, trial.sensor_on)


def repair(chrom: Chromosome, env: Environment, mode: str = cfg.CONSTRAINT_MODE) -> Chromosome:
    """Apply the constraint handling chosen by ``mode`` to ``chrom`` (in place)."""
    clip_genes(chrom, env)                                 # always: genes back into their ranges
    if mode in ("repair", "both"):
        for seg in chrom.segments:
            move_waypoints_out_of_nofly(seg, env)          # 2a: waypoints out of no-fly zones
            fix_crossing_legs(seg, env)                    # 2b: legs around no-fly zones
            trim_to_limits(seg, env)                       # 3 : range and battery
    chrom.forget_fitness()                                 # genes changed -> old fitness invalid
    return chrom

In [ ]:
evaluator = FitnessEvaluator(env)
raw = Chromosome.random(env, np.random.default_rng(1))
evaluator.evaluate(raw, 0)          # without repair
fixed = repair(raw.copy(), env)     # with repair
evaluator.evaluate(fixed, 0)
for name, c in [("before repair", raw), ("after repair", fixed)]:
    o = c.objectives
    print(f"{name:14s} fitness {c.fitness:7.3f} | no-fly share {o['nofly_fraction']:.3f} | "
          f"range overrun {o['range_overrun']:.2f} | battery overrun {o['battery_overrun']:.2f}")

## 6. Genetic operators (`operators.py`)

In [ ]:
"""
Genetic operators: selection, crossover and mutation.

All operators are simple functions. They receive a NumPy random generator
(``rng``) so that a run with the same seed always gives the same result.
"""

import numpy as np



# ---------------------------------------------------------------------------
# Selection: which individuals become parents?
# ---------------------------------------------------------------------------
def tournament_selection(fitness: np.ndarray, n_parents: int, k: int,
                         rng: np.random.Generator) -> np.ndarray:
    """Pick k random individuals and keep the best one; repeat n_parents times.

    k is the SELECTION PRESSURE: with k = 2 weak individuals still win quite
    often (more exploration), with k = 8 the best individuals win almost
    every tournament (more exploitation, risk of premature convergence).
    """
    # one row per tournament, k random individuals in each row
    contestants = rng.integers(0, len(fitness), size=(n_parents, k))
    winners = np.argmax(fitness[contestants], axis=1)      # column of the fittest in each row
    return contestants[np.arange(n_parents), winners]      # index of each tournament winner


def rank_selection(fitness: np.ndarray, n_parents: int, pressure: float,
                   rng: np.random.Generator) -> np.ndarray:
    """Linear ranking: the chance depends on the RANK, not the raw fitness.

    Worst rank gets (2 - pressure) / N, best rank gets pressure / N.
    This keeps the selection pressure constant even when all fitness values
    are very close together (late in the run).
    """
    n = len(fitness)
    ranks = np.empty(n)
    ranks[np.argsort(fitness)] = np.arange(n)          # 0 = worst, n-1 = best
    # probability grows in a straight line from the worst rank to the best rank
    probs = (2 - pressure + 2 * (pressure - 1) * ranks / max(1, n - 1)) / n
    return rng.choice(n, size=n_parents, p=probs / probs.sum())   # draw parents with these chances


def roulette_selection(fitness: np.ndarray, n_parents: int,
                       rng: np.random.Generator) -> np.ndarray:
    """Fitness-proportionate ("roulette wheel") selection.

    Our fitness can be negative, so we first shift it so the worst
    individual has a small positive weight.
    """
    weights = fitness - fitness.min() + 1e-6               # shift so every weight is > 0
    # slice of the "wheel" = weight / total weight
    return rng.choice(len(fitness), size=n_parents, p=weights / weights.sum())


def select_parents(fitness: np.ndarray, n_parents: int, rng: np.random.Generator,
                   method: str = cfg.SELECTION, k: int = cfg.TOURNAMENT_K) -> np.ndarray:
    """Call the selection method chosen in the configuration."""
    if method == "tournament":
        return tournament_selection(fitness, n_parents, k, rng)
    if method == "rank":
        return rank_selection(fitness, n_parents, cfg.RANK_PRESSURE, rng)
    if method == "roulette":
        return roulette_selection(fitness, n_parents, rng)
    raise ValueError(f"unknown selection method: {method}")


# ---------------------------------------------------------------------------
# Crossover: mix two parents into two children
# ---------------------------------------------------------------------------
def blx_alpha(x: np.ndarray, y: np.ndarray, alpha: float, rng: np.random.Generator) -> np.ndarray:
    """BLX-alpha blend crossover for real numbers.

    The child value is drawn uniformly from the interval spanned by the two
    parents, widened by alpha * |x - y| on both sides. For waypoints this is
    a GEOMETRIC crossover: the child waypoint lands in the rectangle between
    the parents' waypoints (or slightly outside it, which keeps diversity).
    """
    low, high = np.minimum(x, y), np.maximum(x, y)         # interval between the two parents
    spread = high - low
    # widen the interval by alpha * spread on each side, then draw uniformly inside it
    return rng.uniform(low - alpha * spread, high + alpha * spread)


def blend_segments(a: DroneSegment, b: DroneSegment, rng: np.random.Generator) -> tuple[DroneSegment, DroneSegment]:
    """Blend two drone segments gene by gene (positions 0..min(Ka, Kb) - 1).

    * waypoints, speeds, powers -> BLX-alpha (real genes)
    * sensor bits               -> uniform crossover (each bit from either parent)
    Extra waypoints of the longer parent are kept unchanged.
    """
    child_a, child_b = a.copy(), b.copy()                  # copies: parents stay unchanged
    n = min(a.n_waypoints, b.n_waypoints)                  # genes that exist in BOTH parents
    for child in (child_a, child_b):                       # each child gets its own random blend
        child.waypoints[:n] = blx_alpha(a.waypoints[:n], b.waypoints[:n], cfg.BLX_ALPHA, rng)
        child.speeds[:n] = blx_alpha(a.speeds[:n], b.speeds[:n], cfg.BLX_ALPHA, rng)
        child.powers[:n] = blx_alpha(a.powers[:n], b.powers[:n], cfg.BLX_ALPHA, rng)
    take_from_b = rng.random(n) < 0.5                     # uniform crossover mask
    child_a.sensor_on[:n] = np.where(take_from_b, b.sensor_on[:n], a.sensor_on[:n])
    child_b.sensor_on[:n] = np.where(take_from_b, a.sensor_on[:n], b.sensor_on[:n])
    return child_a, child_b


def crossover(parent1: Chromosome, parent2: Chromosome, rng: np.random.Generator) -> tuple[Chromosome, Chromosome]:
    """Two-level crossover designed for the multi-segment chromosome.

    Level 1 - drone-segment swap: each drone's whole route is taken from
              parent 1 or parent 2. Good routes are kept intact (a route is
              a "building block" that a cut in the middle could destroy).
    Level 2 - for each drone, with probability 0.5, the two versions are
              blended (BLX-alpha on coordinates, uniform on sensor bits).
    """
    segs1 = [s.copy() for s in parent1.segments]           # child 1 starts as a copy of parent 1
    segs2 = [s.copy() for s in parent2.segments]           # child 2 starts as a copy of parent 2
    for d in range(len(segs1)):                            # for every drone d
        if rng.random() < cfg.SEGMENT_SWAP_PROB:
            segs1[d], segs2[d] = segs2[d], segs1[d]        # level 1: swap drone d's whole route
        if rng.random() < 0.5:
            segs1[d], segs2[d] = blend_segments(segs1[d], segs2[d], rng)   # level 2: blend genes
    return Chromosome(segs1), Chromosome(segs2)


# ---------------------------------------------------------------------------
# Mutation: small random changes that keep the population diverse
# ---------------------------------------------------------------------------
def annealed_sigma(generation: int, total_generations: int) -> float:
    """Step size of the waypoint mutation: large at the start, small at the end.

    sigma(g) = SIGMA_START * (SIGMA_END / SIGMA_START) ** (g / G)
    Big steps early let drones explore the whole map; small steps late
    fine-tune good routes (like simulated annealing).
    """
    progress = min(1.0, generation / max(1, total_generations))   # 0 at the start, 1 at the end
    return cfg.SIGMA_START * (cfg.SIGMA_END / cfg.SIGMA_START) ** progress   # 8 -> 1 cells


def mutate_segment(seg: DroneSegment, rate: float, sigma: float, env: Environment,
                   rng: np.random.Generator) -> DroneSegment:
    """Apply every mutation type to one drone segment (each gene with probability ``rate``)."""
    if rng.random() < cfg.RESET_PROB:                       # rare: brand-new random route
        return DroneSegment.random(env, rng)
    k = seg.n_waypoints
    # Pattern used below: `rng.random(k) < rate` gives a True/False mask that
    # selects each gene with probability `rate`; only the selected genes change.

    # 1) Gaussian perturbation of waypoint coordinates (spatial mutation)
    moved = rng.random(k) < rate
    seg.waypoints[moved] += rng.normal(0.0, sigma, size=(moved.sum(), 2))
    # 2) speed jitter and 3) sensor power jitter
    changed = rng.random(k) < rate
    seg.speeds[changed] += rng.normal(0.0, 0.1 * (cfg.V_MAX - cfg.V_MIN), size=changed.sum())
    changed = rng.random(k) < rate
    seg.powers[changed] += rng.normal(0.0, 0.15, size=changed.sum())
    # 4) sensor bit flip (changes the activation schedule)
    flipped = rng.random(k) < rate
    seg.sensor_on[flipped] = ~seg.sensor_on[flipped]
    # 5) launch from a different base station (integer gene)
    if rng.random() < rate / 2:
        seg.base = int(rng.integers(0, len(env.base_stations)))

    # 6) structural mutations: insert / delete / swap waypoints
    if rng.random() < rate and seg.n_waypoints < cfg.K_MAX:
        i = int(rng.integers(0, seg.n_waypoints))          # insert after waypoint i
        route = seg.route(env)                             # route[i + 1] is waypoint i (route[0] = base)
        # new point = middle of the next leg + a little noise, it copies waypoint i's settings
        new_point = (route[i + 1] + route[i + 2]) / 2 + rng.normal(0.0, sigma, 2)
        seg.waypoints = np.insert(seg.waypoints, i + 1, new_point, axis=0)
        seg.speeds = np.insert(seg.speeds, i + 1, seg.speeds[i])
        seg.powers = np.insert(seg.powers, i + 1, seg.powers[i])
        seg.sensor_on = np.insert(seg.sensor_on, i + 1, seg.sensor_on[i])
    if rng.random() < rate and seg.n_waypoints > cfg.K_MIN:
        i = int(rng.integers(0, seg.n_waypoints))          # delete waypoint i (all 5 of its genes)
        seg.waypoints = np.delete(seg.waypoints, i, axis=0)
        seg.speeds, seg.powers = np.delete(seg.speeds, i), np.delete(seg.powers, i)
        seg.sensor_on = np.delete(seg.sensor_on, i)
    if rng.random() < rate and seg.n_waypoints >= 2:
        i, j = rng.choice(seg.n_waypoints, size=2, replace=False)
        seg.waypoints[[i, j]] = seg.waypoints[[j, i]]      # visit order changes
    return seg


def mutate(chrom: Chromosome, rate: float, sigma: float, env: Environment,
           rng: np.random.Generator) -> Chromosome:
    """Mutate every drone segment of a chromosome (in place)."""
    chrom.segments = [mutate_segment(s, rate, sigma, env, rng) for s in chrom.segments]
    chrom.forget_fitness()                                 # genes changed -> must be re-evaluated
    return chrom

## 7. The Genetic Algorithm engine (`ga.py`)

In [ ]:
"""
The Genetic Algorithm engine.

One generation:
    1. score every individual (fitness uses the penalty of THIS generation)
    2. record statistics (best, mean, std, diversity, objectives, ...)
    3. stop if a termination criterion fires
    4. copy the elite (top ELITISM %) unchanged
    5. fill the rest: select parents -> crossover -> mutate -> repair -> evaluate
    6. if diversity is too low: add random immigrants and boost mutation
"""

import time

import numpy as np



def population_diversity(population: list[Chromosome], env: Environment) -> float:
    """Mean distance between all pairs of individuals (0 = everyone identical).

    Each chromosome is turned into a fixed-length feature vector (its routes
    resampled to 10 points each, see chromosome.path_features); the mean of
    all pairwise Euclidean distances is the diversity.
    """
    features = np.array([path_features(c, env) for c in population])   # one row per individual
    diff = features[:, None, :] - features[None, :, :]     # every row minus every other row
    distances = np.sqrt((diff ** 2).sum(axis=2))           # n x n table of Euclidean distances
    n = len(population)
    return float(distances.sum() / (n * (n - 1)))          # mean over all pairs (diagonal is 0)


class GeneticAlgorithm:
    """Evolves fleet plans. Every setting defaults to the value in config.py."""

    def __init__(self, env: Environment, seed: int = cfg.SEED, pop_size: int = cfg.POP_SIZE,
                 generations: int = cfg.GENERATIONS, mutation_rate: float = cfg.MUTATION_RATE,
                 crossover_rate: float = cfg.CROSSOVER_RATE, selection: str = cfg.SELECTION,
                 tournament_k: int = cfg.TOURNAMENT_K, constraint_mode: str = cfg.CONSTRAINT_MODE,
                 stagnation_patience: int = cfg.STAGNATION_PATIENCE,
                 keep_populations: bool = False, verbose: bool = True) -> None:
        self.env = env
        self.seed = seed
        self.rng = np.random.default_rng(seed)       # ALL randomness of the run comes from here
        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.selection = selection
        self.tournament_k = tournament_k
        self.constraint_mode = constraint_mode
        self.stagnation_patience = stagnation_patience
        self.n_elite = max(1, int(round(cfg.ELITISM * pop_size)))   # e.g. 5 % of 100 = 5 elites
        self.keep_populations = keep_populations
        self.verbose = verbose
        self.evaluator = FitnessEvaluator(env, generations, constraint_mode)

        self.history: list[dict] = []            # one row of statistics per generation
        self.best_per_generation: list[Chromosome] = []  # used for the path / coverage plots
        self.populations: list[list[Chromosome]] = []   # only if keep_populations=True
        self.best: Chromosome | None = None
        self.stop_reason = ""
        self.runtime_s = 0.0

    # ------------------------------------------------------------------
    def random_individual(self, generation: int) -> Chromosome:
        """Create, repair and evaluate one random chromosome."""
        chrom = repair(Chromosome.random(self.env, self.rng), self.env, self.constraint_mode)
        self.evaluator.evaluate(chrom, generation)
        return chrom

    def make_child_pairs(self, population: list[Chromosome], fitness: np.ndarray,
                         n_children: int, rate: float, sigma: float, generation: int) -> list[Chromosome]:
        """Selection -> crossover -> mutation -> repair -> evaluation."""
        # +1 parent so that an odd number of children still has a full last pair
        parents = select_parents(fitness, n_children + 1, self.rng, self.selection, self.tournament_k)
        children = []
        for i in range(0, n_children, 2):                  # two parents -> two children
            p1, p2 = population[parents[i]], population[parents[i + 1]]
            if self.rng.random() < self.crossover_rate:    # usually (90 %) mix the parents ...
                c1, c2 = crossover(p1, p2, self.rng)
            else:                                          # ... otherwise just copy them
                c1, c2 = p1.copy(), p2.copy()
            for child in (c1, c2):
                mutate(child, rate, sigma, self.env, self.rng)
                repair(child, self.env, self.constraint_mode)   # make the child feasible again
                self.evaluator.evaluate(child, generation)
                children.append(child)
        return children[:n_children]                       # drop the extra child if n_children is odd

    def record(self, generation: int, population: list[Chromosome], diversity: float,
               rate: float, sigma: float, immigrants: int, start_time: float) -> None:
        """Save one row of statistics for this generation."""
        fitness = np.array([c.fitness for c in population])
        best = population[int(np.argmax(fitness))]
        # fitness with the LAST generation's penalty, so curves from different
        # generations (and random search) can be compared on the same scale
        final_scale = [self.evaluator.score(c.objectives, self.generations) for c in population]
        obj = best.objectives
        self.history.append({
            "generation": generation,
            "evaluations": self.evaluator.n_evaluations,
            "best_fitness": float(fitness.max()),
            "mean_fitness": float(fitness.mean()),
            "std_fitness": float(fitness.std()),
            "worst_fitness": float(fitness.min()),
            "best_fitness_final_penalty": float(max(final_scale)),
            "diversity": diversity,
            "mutation_rate": rate,
            "sigma": sigma,
            "penalty_lambda": self.evaluator.penalty_factor(generation),
            "immigrants": immigrants,
            "feasible_share": float(np.mean([c.violation < 1e-9 for c in population])),
            "coverage": obj["coverage"], "detection": obj["detection"], "energy": obj["energy"],
            "time": obj["time"], "hazard": obj["hazard"], "violation": best.violation,
            "elapsed_s": time.time() - start_time,
        })
        self.best_per_generation.append(best.copy())
        if self.keep_populations:
            self.populations.append([c.copy() for c in population])

    # ------------------------------------------------------------------
    def run(self) -> Chromosome:
        """Run the GA until a termination criterion fires; return the best plan."""
        start = time.time()
        # generation 0: a completely random (but repaired) population
        population = [self.random_individual(0) for _ in range(self.pop_size)]
        diversity_start = population_diversity(population, self.env)   # reference for "low diversity"
        best_so_far, last_improvement = -np.inf, 0         # used by the stagnation criterion
        rate, immigrants = self.mutation_rate, 0

        for generation in range(self.generations + 1):
            # 1) re-score: the penalty factor depends on the generation number
            for chrom in population:
                self.evaluator.evaluate(chrom, generation)
            fitness = np.array([c.fitness for c in population])
            diversity = population_diversity(population, self.env)
            sigma = annealed_sigma(generation, self.generations)
            self.record(generation, population, diversity, rate, sigma, immigrants, start)
            if self.verbose and (generation % 20 == 0 or generation == self.generations):
                h = self.history[-1]
                print(f"  gen {generation:4d} | best {h['best_fitness']:.4f} | mean {h['mean_fitness']:.4f}"
                      f" | diversity {diversity:.3f} | evals {h['evaluations']}")

            # 2) termination criteria (the reason is stored and printed)
            if fitness.max() > best_so_far + cfg.STAGNATION_EPS:   # a real improvement?
                best_so_far, last_improvement = fitness.max(), generation
            if generation >= self.generations:
                self.stop_reason = f"max generations ({self.generations}) reached"
                break
            if cfg.TARGET_FITNESS is not None and fitness.max() >= cfg.TARGET_FITNESS:
                self.stop_reason = f"target fitness {cfg.TARGET_FITNESS} reached"
                break
            if generation - last_improvement >= self.stagnation_patience:
                self.stop_reason = (f"stagnation: no improvement > {cfg.STAGNATION_EPS} "
                                    f"for {self.stagnation_patience} generations")
                break

            # 3) diversity control: low diversity -> immigrants + stronger mutation
            low_diversity = diversity < cfg.DIVERSITY_THRESHOLD * diversity_start
            rate = self.mutation_rate * (cfg.MUTATION_BOOST if low_diversity else 1.0)
            immigrants = int(round(cfg.IMMIGRANT_FRACTION * self.pop_size)) if low_diversity else 0

            # 4) elitism + offspring
            order = np.argsort(fitness)[::-1]              # best individual first
            elite = [population[i].copy() for i in order[:self.n_elite]]   # survive unchanged
            n_children = self.pop_size - self.n_elite - immigrants          # keep the size constant
            children = self.make_child_pairs(population, fitness, n_children, rate, sigma, generation + 1)
            newcomers = [self.random_individual(generation + 1) for _ in range(immigrants)]
            population = elite + children + newcomers      # the next generation

        final_fitness = np.array([c.fitness for c in population])
        self.best = population[int(np.argmax(final_fitness))].copy()
        self.final_population = population
        self.runtime_s = time.time() - start
        if self.verbose:
            print(f"  stopped: {self.stop_reason} | runtime {self.runtime_s:.1f} s")
        return self.best

## 8. Baseline: random search (`baseline.py`)

In [ ]:
"""
Baseline: pure random search.

Random search creates completely random fleet plans (with the SAME random
generator and the SAME repair as the GA's first generation) and simply keeps
the best one. It gets exactly the same number of fitness evaluations as the
GA, so any difference in quality comes from the GA's selection, crossover
and mutation, not from a bigger budget.
"""

import time

import numpy as np



def random_search(env: Environment, n_evaluations: int, seed: int,
                  generations: int = cfg.GENERATIONS) -> tuple[Chromosome, list[float], float]:
    """Evaluate ``n_evaluations`` random plans and return the best one.

    The fitness is computed with the penalty factor of the GA's LAST
    generation, which is the scale on which the final GA solution is judged.

    Returns (best chromosome, best-so-far fitness after every evaluation, runtime in s).
    """
    start = time.time()
    rng = np.random.default_rng(seed)
    evaluator = FitnessEvaluator(env, generations, cfg.CONSTRAINT_MODE)
    best, best_so_far = None, []
    for _ in range(n_evaluations):
        # exactly like the GA's generation 0: random plan + the same repair
        chrom = repair(Chromosome.random(env, rng), env, cfg.CONSTRAINT_MODE)
        evaluator.evaluate(chrom, generations)             # penalty of the GA's final generation
        if best is None or chrom.fitness > best.fitness:   # keep the best plan seen so far
            best = chrom
        best_so_far.append(best.fitness)                   # for the "best-so-far vs evaluations" plot
    return best, best_so_far, time.time() - start

## 9. Plots (`visualization.py`)

In [ ]:
"""
All plots of the project (matplotlib only).

Map conventions used in every figure:
    * orange shading       hazard intensity (wind + radiation), 0..1
    * dark grey hatched    no-fly zones
    * black dashed line    border of high-priority disaster zones (priority >= 0.5)
    * circles              targets (filled = detected with probability >= 0.5)
    * black triangles      base stations
    * coloured lines       drone routes; SOLID = sensor ON, DASHED = sensor OFF
"""

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


DRONE_COLOURS = ["#2a78d6", "#6250d6", "#008300", "#d55181", "#eb6834", "#1baf7a"]
SERIES_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#d55181", "#6250d6"]
NOFLY_COLOUR = "#3d3d3d"

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.grid": False,
                     "axes.titlesize": 11, "axes.labelsize": 10, "legend.fontsize": 8})


def _save(fig: plt.Figure, path: str) -> None:
    """Save a figure as PNG and free its memory."""
    fig.tight_layout()                    # avoid overlapping labels
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# Maps
# ---------------------------------------------------------------------------
def draw_environment(ax: plt.Axes, env: Environment, show_hazard: bool = True) -> None:
    """Draw hazard field, no-fly zones, priority zones and base stations."""
    extent = (0, env.size, 0, env.size)   # image covers x = 0..100 and y = 0..100
    if show_hazard:                       # vmax 1.3 keeps the strongest hazard readable under the routes
        ax.imshow(env.hazard, origin="lower", extent=extent, cmap="Oranges", vmin=0, vmax=1.3, alpha=0.9)
    # masked array: only the blocked cells are drawn, the others stay transparent
    nofly = np.ma.masked_where(~env.blocked, env.blocked.astype(float))
    ax.imshow(nofly, origin="lower", extent=extent, cmap=ListedColormap([NOFLY_COLOUR]), alpha=0.9)
    ax.contourf(env.cell_centres_x, env.cell_centres_x, env.blocked.astype(float), levels=[0.5, 1.5],
                colors="none", hatches=["////"])
    # dashed line = border of the high-priority disaster zones (priority = 0.5)
    ax.contour(env.cell_centres_x, env.cell_centres_x, env.priority, levels=[0.5],
               colors="black", linestyles="--", linewidths=1.0)
    ax.scatter(env.base_stations[:, 0], env.base_stations[:, 1], marker="^", s=110,
               c="black", edgecolors="white", linewidths=1.2, zorder=6)
    ax.set_xlim(0, env.size); ax.set_ylim(0, env.size)
    ax.set_aspect("equal")
    ax.set_xlabel("x (cells, 10 m)"); ax.set_ylabel("y (cells, 10 m)")


def draw_routes(ax: plt.Axes, chrom: Chromosome, env: Environment, show_targets: bool = True) -> None:
    """Draw every drone route; solid legs = sensor on, dashed legs = sensor off."""
    for d, seg in enumerate(chrom.segments):
        colour = DRONE_COLOURS[d % len(DRONE_COLOURS)]
        route = seg.route(env)
        sensor = np.append(seg.sensor_on, False)          # return leg: sensor off
        for k in range(len(route) - 1):                   # draw leg k: route[k] -> route[k + 1]
            ax.plot(route[k:k + 2, 0], route[k:k + 2, 1], color=colour,
                    linestyle="-" if sensor[k] else "--",
                    linewidth=2.2 if sensor[k] else 1.1, zorder=4)
        ax.scatter(seg.waypoints[:, 0], seg.waypoints[:, 1], s=12, color=colour, zorder=5)
    if show_targets:
        p = detection_probabilities(build_legs(chrom, env), env)
        found = p >= 0.5                                  # "detected" = probability of at least 50 %
        ax.scatter(env.targets[found, 0], env.targets[found, 1], s=28, c="black",
                   edgecolors="white", linewidths=0.8, zorder=7)
        ax.scatter(env.targets[~found, 0], env.targets[~found, 1], s=28, facecolors="white",
                   edgecolors="black", linewidths=1.0, zorder=7)


def legend_handles(n_drones: int, with_routes: bool = True) -> list:
    """Legend entries for the map symbols (route entries only if routes are drawn)."""
    handles = [Line2D([], [], color=DRONE_COLOURS[d], lw=2.2, label=f"Drone {d + 1}") for d in range(n_drones)]
    if with_routes:
        handles += [Line2D([], [], color="grey", lw=2.2, label="sensor ON"),
                    Line2D([], [], color="grey", lw=1.1, ls="--", label="sensor OFF")]
    handles += [
        Patch(facecolor=NOFLY_COLOUR, hatch="////", label="no-fly zone"),
        Patch(facecolor="#f4a261", label="hazard (darker = worse)"),
        Line2D([], [], color="black", ls="--", label="priority zone (P >= 0.5)"),
    ]
    if with_routes:
        handles += [Line2D([], [], marker="o", ls="", mfc="black", mec="black", label="target detected"),
                    Line2D([], [], marker="o", ls="", mfc="white", mec="black", label="target missed")]
    else:
        handles += [Line2D([], [], marker="o", ls="", mfc="white", mec="black", label="target")]
    handles += [Line2D([], [], marker="^", ls="", mfc="black", mec="black", ms=8, label="base station")]
    return handles


def map_legend(ax: plt.Axes, n_drones: int) -> None:
    """Legend to the RIGHT of a single map, so it never hides part of the map."""
    ax.legend(handles=legend_handles(n_drones), loc="upper left", bbox_to_anchor=(1.01, 1.0), framealpha=0.95)


def figure_legend(fig: plt.Figure, n_drones: int, with_routes: bool = True) -> None:
    """One shared legend BELOW a grid of maps."""
    fig.legend(handles=legend_handles(n_drones, with_routes), loc="upper center",
               bbox_to_anchor=(0.5, 0.0), ncol=6, framealpha=0.95)


def short_stats(obj: dict, fitness: float) -> str:
    return (f"F={fitness:.3f} | cov={obj['coverage']:.2f} det={obj['detection']:.2f} "
            f"E={obj['energy']:.2f} T={obj['time']:.2f} H={obj['hazard']:.2f}")


def plot_environment(env: Environment, path: str) -> None:
    """The map on its own: hazard, priority map, no-fly zones, targets."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    draw_environment(axes[0], env)
    axes[0].scatter(env.targets[:, 0], env.targets[:, 1], s=22, facecolors="white", edgecolors="black", zorder=7)
    axes[0].set_title("Hazard field, no-fly zones, targets, bases")
    im = axes[1].imshow(env.priority, origin="lower", extent=(0, env.size, 0, env.size), cmap="Blues", vmin=0, vmax=1)
    nofly = np.ma.masked_where(~env.blocked, env.blocked.astype(float))
    axes[1].imshow(nofly, origin="lower", extent=(0, env.size, 0, env.size), cmap=ListedColormap([NOFLY_COLOUR]))
    axes[1].set_title("Priority map (disaster zones)")
    axes[1].set_xlabel("x (cells, 10 m)"); axes[1].set_ylabel("y (cells, 10 m)")
    fig.colorbar(im, ax=axes[1], label="priority weight")
    figure_legend(fig, 0, with_routes=False)
    _save(fig, path)


def plot_routes_over_generations(snapshots: list[tuple[int, Chromosome]], env: Environment, path: str) -> None:
    """2 x 2 grid: best fleet plan at four generations."""
    fig, axes = plt.subplots(2, 2, figsize=(11, 11))
    for ax, (gen, chrom) in zip(axes.ravel(), snapshots):
        draw_environment(ax, env)
        draw_routes(ax, chrom, env)
        ax.set_title(f"Generation {gen}\n{short_stats(chrom.objectives, chrom.fitness)}", fontsize=9)
    figure_legend(fig, len(snapshots[0][1].segments))
    fig.suptitle("Best UAV flight plan across generations", fontsize=13)
    _save(fig, path)


def plot_coverage_heatmaps(snapshots: list[tuple[int, Chromosome, list[Chromosome]]], env: Environment, path: str) -> None:
    """Row 1: priority-weighted coverage of the best plan.
    Row 2: share of the whole population whose sensors cover each cell."""
    fig, axes = plt.subplots(2, len(snapshots), figsize=(4.2 * len(snapshots), 8.4))
    for col, (gen, best, population) in enumerate(snapshots):
        covered = coverage_map(best, env)
        shown = np.where(covered, env.priority, np.nan)   # NaN = not covered -> drawn white
        ax = axes[0, col]
        ax.imshow(np.zeros_like(env.priority), origin="lower", extent=(0, env.size, 0, env.size), cmap="Greys", vmin=0, vmax=1)
        im1 = ax.imshow(shown, origin="lower", extent=(0, env.size, 0, env.size), cmap="viridis", vmin=0, vmax=1)
        draw_environment(ax, env, show_hazard=False)
        ax.set_title(f"Gen {gen}: best plan coverage\n(priority-weighted = {best.objectives['coverage']:.2f})", fontsize=9)
        # mean of True/False maps = share of individuals that cover each cell
        freq = np.mean([coverage_map(c, env) for c in population], axis=0)
        ax = axes[1, col]
        im2 = ax.imshow(freq, origin="lower", extent=(0, env.size, 0, env.size), cmap="magma", vmin=0, vmax=1)
        draw_environment(ax, env, show_hazard=False)
        ax.set_title(f"Gen {gen}: population coverage frequency", fontsize=9)
        if col > 0:                       # only the first column needs y-axis labels
            axes[0, col].set_ylabel(""); axes[1, col].set_ylabel("")
    fig.colorbar(im1, ax=axes[0, :].tolist(), shrink=0.8, label="priority of covered cell (white = not covered)")
    fig.colorbar(im2, ax=axes[1, :].tolist(), shrink=0.8, label="share of population covering the cell")
    fig.suptitle("Coverage heatmaps across generations", fontsize=13)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_final_solution(chrom: Chromosome, env: Environment, path: str, title: str) -> None:
    """One large, annotated map of a single fleet plan."""
    fig, ax = plt.subplots(figsize=(10, 10))
    draw_environment(ax, env)
    draw_routes(ax, chrom, env)
    for d, seg in enumerate(chrom.segments):
        ax.annotate(f"D{d + 1}", seg.waypoints[0], xytext=(4, 4), textcoords="offset points",
                    color=DRONE_COLOURS[d], fontsize=10, fontweight="bold")
    o = chrom.objectives
    ax.set_title(f"{title}\nfitness {chrom.fitness:.3f} | coverage {o['coverage']:.2f} | detected "
                 f"{o['targets_detected']}/{len(env.targets)} targets | energy {o['energy_kj']:.0f} kJ | "
                 f"makespan {o['makespan_s']:.0f} s | hazard {o['hazard']:.2f}", fontsize=10)
    map_legend(ax, len(chrom.segments))
    _save(fig, path)


# ---------------------------------------------------------------------------
# Curves
# ---------------------------------------------------------------------------
def plot_convergence(history, path: str) -> None:
    """Best and average fitness per generation (average shown with +/- 1 std band)."""
    g = history["generation"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.fill_between(g, history["mean_fitness"] - history["std_fitness"],
                    history["mean_fitness"] + history["std_fitness"], color=SERIES_COLOURS[1], alpha=0.18,
                    label="average +/- 1 std")
    ax.plot(g, history["mean_fitness"], color=SERIES_COLOURS[1], lw=2, label="average fitness")
    ax.plot(g, history["best_fitness"], color=SERIES_COLOURS[0], lw=2, label="best fitness")
    ax.set_xlabel("generation"); ax.set_ylabel("fitness F")
    ax.set_title("GA convergence: best vs. average fitness")
    ax.grid(alpha=0.3); ax.legend(loc="lower right")
    _save(fig, path)


def plot_diversity(history, path: str, threshold: float | None = None) -> None:
    """Population diversity, with generations that received immigrants marked."""
    fig, axes = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
    axes[0].plot(history["generation"], history["diversity"], color=SERIES_COLOURS[0], lw=2)
    boosted = history[history["immigrants"] > 0]
    axes[0].scatter(boosted["generation"], boosted["diversity"], color=SERIES_COLOURS[1], s=14,
                    zorder=3, label="immigrants added + mutation boosted")
    if threshold is not None:             # below this line immigrants would be added
        axes[0].axhline(threshold, color="grey", ls="--", lw=1.2, label="immigrant threshold (30 % of start)")
    axes[0].set_ylabel("mean pairwise distance"); axes[0].set_title("Population diversity")
    axes[0].grid(alpha=0.3)
    axes[0].legend(loc="upper right")
    axes[1].plot(history["generation"], history["sigma"], color=SERIES_COLOURS[2], lw=2)
    axes[1].set_ylabel("mutation step sigma (cells)"); axes[1].set_xlabel("generation")
    axes[1].set_title("Annealed waypoint mutation step"); axes[1].grid(alpha=0.3)
    _save(fig, path)


def plot_objectives(history, path: str) -> None:
    """Each raw objective of the best individual over the generations (small multiples)."""
    names = [("coverage", "Coverage (max)"), ("detection", "Detection rate (max)"),
             ("energy", "Energy (min)"), ("time", "Flight time (min)"),
             ("hazard", "Hazard exposure (min)"), ("violation", "Constraint violation (min)")]
    fig, axes = plt.subplots(2, 3, figsize=(11, 5.5), sharex=True)
    for ax, (key, label) in zip(axes.ravel(), names):
        ax.plot(history["generation"], history[key], color=SERIES_COLOURS[0], lw=2)
        ax.set_title(label); ax.grid(alpha=0.3)
    for ax in axes[1]:
        ax.set_xlabel("generation")
    for ax in axes[:, 0]:
        ax.set_ylabel("normalised value")
    fig.suptitle("Objectives of the best individual per generation", fontsize=12)
    _save(fig, path)


def plot_sweep(curves: dict, title: str, path: str, x_label: str = "generation") -> None:
    """Mean best-fitness curve (+/- std over seeds) for each setting of a sweep."""
    fig, ax = plt.subplots(figsize=(7, 4.3))
    for i, (label, runs) in enumerate(curves.items()):
        length = min(len(r) for r in runs)               # runs can differ in length; cut to the shortest
        data = np.array([r[:length] for r in runs])       # one row per seed
        mean, std = data.mean(axis=0), data.std(axis=0)   # mean and spread over the seeds
        x = np.arange(length)
        colour = SERIES_COLOURS[i % len(SERIES_COLOURS)]
        ax.plot(x, mean, color=colour, lw=2, label=label)
        ax.fill_between(x, mean - std, mean + std, color=colour, alpha=0.15)
    ax.set_xlabel(x_label); ax.set_ylabel("best fitness (mean of seeds)")
    ax.set_title(title); ax.grid(alpha=0.3); ax.legend(loc="lower right")
    _save(fig, path)


def plot_ga_vs_random(ga_curves: list, rs_curves: list, path: str) -> None:
    """Best-so-far fitness against the number of fitness evaluations."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for label, runs, colour in (("Genetic Algorithm", ga_curves, SERIES_COLOURS[0]),
                                ("Random search", rs_curves, SERIES_COLOURS[1])):
        length = min(len(r[0]) for r in runs)
        evals = runs[0][0][:length]
        data = np.array([r[1][:length] for r in runs])
        mean, std = data.mean(axis=0), data.std(axis=0)
        ax.plot(evals, mean, color=colour, lw=2, label=f"{label} (mean of {len(runs)} seeds)")
        ax.fill_between(evals, mean - std, mean + std, color=colour, alpha=0.18)
    ax.set_xlabel("fitness evaluations"); ax.set_ylabel("best fitness found so far")
    ax.set_title("GA vs. random search with the same evaluation budget")
    ax.grid(alpha=0.3); ax.legend(loc="lower right")
    _save(fig, path)


def plot_side_by_side(ga_best: Chromosome, rs_best: Chromosome, env: Environment, path: str) -> None:
    """Routes (top) and coverage (bottom) of the GA solution next to the random-search one."""
    fig, axes = plt.subplots(2, 2, figsize=(11, 11))
    for col, (name, chrom) in enumerate((("Genetic Algorithm", ga_best), ("Random search", rs_best))):
        draw_environment(axes[0, col], env)
        draw_routes(axes[0, col], chrom, env)
        axes[0, col].set_title(f"{name}: routes\n{short_stats(chrom.objectives, chrom.fitness)}", fontsize=9)
        covered = np.where(coverage_map(chrom, env), env.priority, np.nan)
        im = axes[1, col].imshow(covered, origin="lower", extent=(0, env.size, 0, env.size),
                                 cmap="viridis", vmin=0, vmax=1)
        draw_environment(axes[1, col], env, show_hazard=False)
        axes[1, col].set_title(f"{name}: covered cells (colour = priority)\n"
                               f"high-priority cells covered: {100 * chrom.objectives['high_priority_coverage']:.0f}%",
                               fontsize=9)
    fig.colorbar(im, ax=axes[1, :].tolist(), shrink=0.7, label="priority of covered cell")
    fig.legend(handles=legend_handles(len(ga_best.segments)), loc="upper center",
               bbox_to_anchor=(0.45, 0.06), ncol=6, framealpha=0.95)
    fig.suptitle("Best GA plan vs. best random-search plan (same seed, same budget)", fontsize=13)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

## 10. Experiments (`main.py`)
These functions run the three experiments and save every figure and table into a results folder.

In [ ]:
"""
Command-line entry point: runs the experiments and saves all results.

Examples
--------
    python main.py                          # main GA run (pop 100, 200 generations, seed 42)
    python main.py --experiment sensitivity # hyper-parameter sweeps
    python main.py --experiment baseline    # GA vs. random search
    python main.py --experiment all         # everything above
    python main.py --experiment all --quick # whole pipeline in ~2 minutes (for testing)
    python main.py --seed 7 --generations 50 --pop-size 60
"""

import argparse
import json
import os
import time

import numpy as np
import pandas as pd


# Settings used by --quick (small numbers, only meant to test the pipeline).
QUICK = {"pop_size": 30, "generations": 20, "sweep_generations": 10, "sweep_seeds": [1],
         "baseline_seeds": [1, 2], "sweep_pop_sizes": [20, 50], "sweep_mutation_rates": [0.05, 0.2],
         "sweep_tournament_k": [2, 5], "sweep_selections": ["roulette", "tournament"],
         "sweep_constraint_modes": ["penalty", "both"]}


def save_json(data: dict, path: str) -> None:
    """Write a dictionary to a JSON file (NumPy numbers are converted to plain numbers)."""
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))


def snapshot_generations(last: int) -> list[int]:
    """Generation 0, ~25 %, ~50 % and the final generation."""
    return [0, round(0.25 * last), round(0.5 * last), last]


# ---------------------------------------------------------------------------
# Experiment 1: the main run
# ---------------------------------------------------------------------------
def run_main(env: Environment, out: str, seed: int, pop_size: int, generations: int) -> None:
    os.makedirs(out, exist_ok=True)
    print(f"\n=== Main GA run: seed {seed}, population {pop_size}, {generations} generations ===")
    ga = GeneticAlgorithm(env, seed=seed, pop_size=pop_size, generations=generations, keep_populations=True)
    best = ga.run()
    history = pd.DataFrame(ga.history)
    history.to_csv(os.path.join(out, "history.csv"), index=False)

    last = int(history["generation"].iloc[-1])
    gens = snapshot_generations(last)
    plot_environment(env, os.path.join(out, "environment.png"))
    plot_convergence(history, os.path.join(out, "convergence.png"))
    plot_diversity(history, os.path.join(out, "diversity.png"),
                       threshold=cfg.DIVERSITY_THRESHOLD * history["diversity"].iloc[0])
    plot_objectives(history, os.path.join(out, "objectives_over_time.png"))
    plot_routes_over_generations([(g, ga.best_per_generation[g]) for g in gens], env,
                                     os.path.join(out, "paths_generations.png"))
    plot_coverage_heatmaps([(g, ga.best_per_generation[g], ga.populations[g]) for g in gens], env,
                               os.path.join(out, "coverage_heatmaps.png"))
    plot_final_solution(best, env, os.path.join(out, "final_solution.png"),
                            f"Final evolved fleet plan (seed {seed}, generation {last})")

    save_json({
        "seed": seed, "pop_size": pop_size, "max_generations": generations,
        "generations_run": last, "stop_reason": ga.stop_reason,
        "fitness_evaluations": ga.evaluator.n_evaluations, "runtime_s": round(ga.runtime_s, 1),
        "initial_best_fitness": history["best_fitness"].iloc[0],
        "initial_mean_fitness": history["mean_fitness"].iloc[0],
        "final_best_fitness": best.fitness, "final_mean_fitness": history["mean_fitness"].iloc[-1],
        "immigrant_generations": int((history["immigrants"] > 0).sum()),
        "best_objectives": best.objectives, "snapshot_generations": gens,
        "best_plan": best.describe(), "config": cfg.as_dict(),
    }, os.path.join(out, "run_summary.json"))
    print(f"Saved main-run results to {out}/")


# ---------------------------------------------------------------------------
# Experiment 2: hyper-parameter sensitivity
# ---------------------------------------------------------------------------
def generations_to_converge(curve: list[float]) -> int:
    """First generation at which 99 % of the total improvement was reached."""
    curve = np.array(curve)
    goal = curve[0] + 0.99 * (curve[-1] - curve[0])        # start + 99 % of the total gain
    return int(np.argmax(curve >= goal))                   # argmax of True/False = first True


def run_sweep(env: Environment, out: str, name: str, parameter: str, values: list,
              seeds: list[int], generations: int, pop_size: int) -> pd.DataFrame:
    """Run the GA for every value of one parameter (all other settings fixed)."""
    rows, curves = [], {}
    for value in values:
        label = f"{parameter} = {value}"
        curves[label] = []
        for seed in seeds:
            # only ONE parameter changes, everything else keeps its default value;
            # stagnation is switched off so every run lasts the same number of generations
            settings = {"pop_size": pop_size, parameter: value}
            ga = GeneticAlgorithm(env, seed=seed, generations=generations, verbose=False,
                                  stagnation_patience=10 ** 9, **settings)
            best = ga.run()
            curve = [h["best_fitness"] for h in ga.history]
            curves[label].append(curve)
            rows.append({"sweep": name, "parameter": parameter, "value": value, "seed": seed,
                         "final_best_fitness": best.fitness, "final_mean_fitness": ga.history[-1]["mean_fitness"],
                         "generations_to_converge": generations_to_converge(curve),
                         "final_diversity": ga.history[-1]["diversity"], "runtime_s": ga.runtime_s,
                         "evaluations": ga.evaluator.n_evaluations, "violation": best.violation,
                         "nofly_fraction": best.objectives["nofly_fraction"],
                         "feasible_share": ga.history[-1]["feasible_share"],
                         "coverage": best.objectives["coverage"], "detection": best.objectives["detection"]})
            print(f"  [{name}] {label:<28} seed {seed}: best {best.fitness:.4f} ({ga.runtime_s:.0f} s)")
    plot_sweep(curves, f"Sensitivity: {name}", os.path.join(out, f"sweep_{parameter}.png"))
    return pd.DataFrame(rows)


def run_sensitivity(env: Environment, out: str, quick: bool) -> None:
    os.makedirs(out, exist_ok=True)
    q = QUICK if quick else {}
    seeds = q.get("sweep_seeds", cfg.SWEEP_SEEDS)
    gens = q.get("sweep_generations", cfg.SWEEP_GENERATIONS)
    pop = q.get("pop_size", cfg.POP_SIZE)
    sweeps = [
        ("population size", "pop_size", q.get("sweep_pop_sizes", cfg.SWEEP_POP_SIZES)),
        ("mutation rate", "mutation_rate", q.get("sweep_mutation_rates", cfg.SWEEP_MUTATION_RATES)),
        ("tournament size k (selection pressure)", "tournament_k", q.get("sweep_tournament_k", cfg.SWEEP_TOURNAMENT_K)),
        ("selection method", "selection", q.get("sweep_selections", cfg.SWEEP_SELECTIONS)),
        ("constraint handling", "constraint_mode", q.get("sweep_constraint_modes", cfg.SWEEP_CONSTRAINT_MODES)),
    ]
    print(f"\n=== Sensitivity analysis: seeds {seeds}, {gens} generations per run ===")
    all_runs = []
    for name, parameter, values in sweeps:
        all_runs.append(run_sweep(env, out, name, parameter, values, seeds, gens, pop))
        pd.concat(all_runs).to_csv(os.path.join(out, "sensitivity_runs.csv"), index=False)  # save progress
    runs = pd.concat(all_runs)
    summary = (runs.groupby(["sweep", "parameter", "value"], sort=False)
               .agg(best_mean=("final_best_fitness", "mean"), best_std=("final_best_fitness", "std"),
                    converge_gen_mean=("generations_to_converge", "mean"),
                    converge_gen_std=("generations_to_converge", "std"),
                    runtime_mean_s=("runtime_s", "mean"), evaluations=("evaluations", "mean"),
                    final_diversity=("final_diversity", "mean"), violation_mean=("violation", "mean"),
                    feasible_share=("feasible_share", "mean"))
               .reset_index())
    summary.to_csv(os.path.join(out, "sensitivity_summary.csv"), index=False)
    print(summary.to_string(index=False))


# ---------------------------------------------------------------------------
# Experiment 3: GA vs. random search
# ---------------------------------------------------------------------------
METRICS = ["fitness", "coverage", "high_priority_coverage", "detection", "targets_detected",
           "energy_kj", "makespan_s", "hazard", "violation"]


def metric_row(method: str, seed: int, chrom, evaluations: int, runtime: float) -> dict:
    o = chrom.objectives
    return {"method": method, "seed": seed, "fitness": chrom.fitness, "coverage": o["coverage"],
            "high_priority_coverage": o["high_priority_coverage"], "detection": o["detection"],
            "targets_detected": o["targets_detected"], "energy_kj": o["energy_kj"],
            "makespan_s": o["makespan_s"], "hazard": o["hazard"], "violation": chrom.violation,
            "evaluations": evaluations, "runtime_s": runtime}


def run_baseline(env: Environment, out: str, seeds: list[int], pop_size: int, generations: int) -> None:
    os.makedirs(out, exist_ok=True)
    print(f"\n=== GA vs. random search: seeds {seeds} ===")
    rows, ga_curves, rs_curves, examples = [], [], [], None
    for seed in seeds:
        ga = GeneticAlgorithm(env, seed=seed, pop_size=pop_size, generations=generations, verbose=False)
        ga_best = ga.run()
        budget = ga.evaluator.n_evaluations                # random search gets exactly this many
        # random search is judged with the penalty of the GA's last generation
        rs_best, rs_curve, rs_time = random_search(env, budget, seed, generations=generations)
        ga_best.fitness = ga.evaluator.score(ga_best.objectives, generations)   # same penalty scale
        rows.append(metric_row("Genetic Algorithm", seed, ga_best, budget, ga.runtime_s))
        rows.append(metric_row("Random search", seed, rs_best, budget, rs_time))
        evals = [h["evaluations"] for h in ga.history]
        # best-so-far curve of the GA (running maximum), sampled once per generation
        ga_curve = np.maximum.accumulate([h["best_fitness_final_penalty"] for h in ga.history])
        ga_curves.append((evals, ga_curve))
        rs_curves.append((evals, [rs_curve[e - 1] for e in evals]))
        if examples is None:
            examples = (ga_best, rs_best, seed)
        print(f"  seed {seed}: GA {ga_best.fitness:.4f} vs random {rs_best.fitness:.4f} "
              f"({budget} evaluations each)")

    runs = pd.DataFrame(rows)
    runs.to_csv(os.path.join(out, "baseline_runs.csv"), index=False)
    table = runs.groupby("method")[METRICS + ["runtime_s"]].agg(["mean", "std"])
    table.columns = [f"{m}_{s}" for m, s in table.columns]
    table.to_csv(os.path.join(out, "baseline_summary.csv"))
    print(table.T.to_string())
    plot_ga_vs_random(ga_curves, rs_curves, os.path.join(out, "ga_vs_random_convergence.png"))
    plot_side_by_side(examples[0], examples[1], env, os.path.join(out, "ga_vs_random_solutions.png"))
    save_json({"seeds": seeds, "example_seed": examples[2], "ga_example_plan": examples[0].describe(),
               "random_example_plan": examples[1].describe()}, os.path.join(out, "baseline_examples.json"))

### 10.1 Main run
Full settings: population 100, 200 generations, seed 42 (about 3–5 minutes on Colab).
Set `MAIN_QUICK = True` for a 20-second test.

In [ ]:
MAIN_QUICK = False
OUT = "results_notebook"
pop, gens = (30, 20) if MAIN_QUICK else (cfg.POP_SIZE, cfg.GENERATIONS)
run_main(env, OUT, seed=cfg.SEED, pop_size=pop, generations=gens)
for name in ["convergence", "diversity", "objectives_over_time", "paths_generations",
             "coverage_heatmaps", "final_solution"]:
    display(Image(f"{OUT}/{name}.png", width=850))

### 10.2 Hyper-parameter sensitivity
The full sweep (3 seeds × 19 settings × 100 generations) takes more than an hour.
`SWEEP_QUICK = True` runs a small version in a few minutes.

In [ ]:
SWEEP_QUICK = True
run_sensitivity(env, f"{OUT}/sensitivity", quick=SWEEP_QUICK)
display(pd.read_csv(f"{OUT}/sensitivity/sensitivity_summary.csv").round(4))
for name in ["pop_size", "mutation_rate", "tournament_k", "selection", "constraint_mode"]:
    display(Image(f"{OUT}/sensitivity/sweep_{name}.png", width=650))

### 10.3 GA vs. random search (same number of fitness evaluations)
`BASELINE_QUICK = True` uses 2 seeds and a small GA; set it to `False` for the full comparison (5 seeds).

In [ ]:
BASELINE_QUICK = True
seeds, pop, gens = ([1, 2], 30, 20) if BASELINE_QUICK else (cfg.BASELINE_SEEDS, cfg.POP_SIZE, cfg.GENERATIONS)
run_baseline(env, f"{OUT}/baseline", seeds, pop, gens)
display(pd.read_csv(f"{OUT}/baseline/baseline_summary.csv", index_col=0).T.round(3))
display(Image(f"{OUT}/baseline/ga_vs_random_convergence.png", width=750))
display(Image(f"{OUT}/baseline/ga_vs_random_solutions.png", width=850))

## Try it yourself
* change a fitness weight, e.g. `cfg.W_HAZARD = 0.4`, and re-run 10.1 – the drones avoid the orange hazard pockets more;
* GA settings are passed as arguments, for example
  `GeneticAlgorithm(env, mutation_rate=0.2, tournament_k=5, constraint_mode="penalty").run()`
  switches repair off and uses a stronger selection pressure – compare the convergence curves.